# ft_linux

The goal of this project is to build a linux distro using the Linux From Scratch (LFS) book. The version I followed was 12.4. Obviously, this repo wont contain the whole project, just finished distro and the very long list of instructions.

Be aware that you may encounter problems, thuis is by no means a perfect solution. Any problems are up to you to solve but there are plenty of resources to help you out

# Steps Taken:

[Creating the Build Environment](#creating-the-build-environment)
- [Escaping the VM](#escaping-the-vm)<br>
- [Disk Partitioning](#disk-partitioning)<br>
- [Setting the LFS Environment](#setting-the-lfs-environment)
- [Getting the Packages](#getting-the-packages)
- [Chroot](#chroot)
- [Set Up Your Environment](#set-up-your-environment)

# Creating the Build Environment.

## Escaping the VM

First of all you wont be building this on your host machine. You're gonna want to build this in its own environment. <br>
The way i did this was to use a ~~Gentoo~~  ~~Arch~~ ~~Gentoo~~ [Debian Live CD](http://cdimage.debian.org/debian-cd/current-live/amd64/iso-hybrid/) in a virtual machine. 

Im using Oracel Virtual Machine which I gave 8GB of RAM , 6 cores (dont know how neccessary this is, I think one is ok) and then 25GB of storage space. For someone who knows what they are doing, these choices might not make sense. But I'm learning by doing.

After booting the VM, you will be in a familiar GNOME desktop environment. We can just skipp the installation process and just open a terminal. Then its just a matter of creating a password and enabling ssh.

To change the password, the command is ```passwd```

In [ ]:
sudo passwd user

Great, now we need to enable ssh

In [ ]:
# if no sshd
sudo apt-get install ssh

systemctl status sshd  # Show SSH Service status, should be enabled
ip addr show          # Show IP address

Now we can immediatley ditch our VM, because it's always better on the Host Machine. We can use ````ssh````to access the Virtual Machine. This way we can easily access documents like the LFS book to assist us in making this project.

In [ ]:
# On Host
ssh user@XXX.XXX.X.X # Double check what the user is called

//TODO: FIX

Before we carry on, we need to make sure our host machine is setup to build some of the packages. LFS provides a script to check some packages and aliases

In [ ]:
cat > version-check.sh << "EOF"
#!/bin/bash
# A script to list version numbers of critical development tools

# If you have tools installed in other directories, adjust PATH here AND
# in ~lfs/.bashrc (section 4.4) as well.

LC_ALL=C 
PATH=/usr/bin:/bin

bail() { echo "FATAL: $1"; exit 1; }
grep --version > /dev/null 2> /dev/null || bail "grep does not work"
sed '' /dev/null || bail "sed does not work"
sort   /dev/null || bail "sort does not work"

ver_check()
{
   if ! type -p $2 &>/dev/null
   then 
     echo "ERROR: Cannot find $2 ($1)"; return 1; 
   fi
   v=$($2 --version 2>&1 | grep -E -o '[0-9]+\.[0-9\.]+[a-z]*' | head -n1)
   if printf '%s\n' $3 $v | sort --version-sort --check &>/dev/null
   then 
     printf "OK:    %-9s %-6s >= $3\n" "$1" "$v"; return 0;
   else 
     printf "ERROR: %-9s is TOO OLD ($3 or later required)\n" "$1"; 
     return 1; 
   fi
}

ver_kernel()
{
   kver=$(uname -r | grep -E -o '^[0-9\.]+')
   if printf '%s\n' $1 $kver | sort --version-sort --check &>/dev/null
   then 
     printf "OK:    Linux Kernel $kver >= $1\n"; return 0;
   else 
     printf "ERROR: Linux Kernel ($kver) is TOO OLD ($1 or later required)\n" "$kver"; 
     return 1; 
   fi
}

# Coreutils first because --version-sort needs Coreutils >= 7.0
ver_check Coreutils      sort     8.1 || bail "Coreutils too old, stop"
ver_check Bash           bash     3.2
ver_check Binutils       ld       2.13.1
ver_check Bison          bison    2.7
ver_check Diffutils      diff     2.8.1
ver_check Findutils      find     4.2.31
ver_check Gawk           gawk     4.0.1
ver_check GCC            gcc      5.4
ver_check "GCC (C++)"    g++      5.4
ver_check Grep           grep     2.5.1a
ver_check Gzip           gzip     1.3.12
ver_check M4             m4       1.4.10
ver_check Make           make     4.0
ver_check Patch          patch    2.5.4
ver_check Perl           perl     5.8.8
ver_check Python         python3  3.4
ver_check Sed            sed      4.1.5
ver_check Tar            tar      1.22
ver_check Texinfo        texi2any 5.0
ver_check Xz             xz       5.0.0
ver_kernel 5.4

if mount | grep -q 'devpts on /dev/pts' && [ -e /dev/ptmx ]
then echo "OK:    Linux Kernel supports UNIX 98 PTY";
else echo "ERROR: Linux Kernel does NOT support UNIX 98 PTY"; fi

alias_check() {
   if $1 --version 2>&1 | grep -qi $2
   then printf "OK:    %-4s is $2\n" "$1";
   else printf "ERROR: %-4s is NOT $2\n" "$1"; fi
}
echo "Aliases:"
alias_check awk GNU
alias_check yacc Bison
alias_check sh Bash

echo "Compiler check:"
if printf "int main(){}" | g++ -x c++ -
then echo "OK:    g++ works";
else echo "ERROR: g++ does NOT work"; fi
rm -f a.out

if [ "$(nproc)" = "" ]; then
   echo "ERROR: nproc is not available or it produces empty output"
else
   echo "OK: nproc reports $(nproc) logical cores are available"
fi
EOF

bash version-check.sh

Make a note of what is not there. for me its:
- bison
- gawk
- m4
- texinfo
- Under ```alias``` sh is not bash

So we use 

In [ ]:
sudo apt-get install bison gawk m5 texinfo # Plus wget

Then to fix the alias of ```sh```:

In [ ]:
sudo ln -sf bash /bin/sh

# Verify change
ls -l /bin/sh # Should see /bin/sh -> bash

We can now check our host machine with our ```version-check.sh```:

In [ ]:
user@debian:~$ sh version-check.sh 
OK:    Coreutils 9.7    >= 8.1
OK:    Bash      5.2.37 >= 3.2
OK:    Binutils  2.44   >= 2.13.1
OK:    Bison     3.8.2  >= 2.7
OK:    Diffutils 3.10   >= 2.8.1
OK:    Findutils 4.10.0 >= 4.2.31
OK:    Gawk      5.2.1  >= 4.0.1
OK:    GCC       14.2.0 >= 5.4
OK:    GCC (C++) 14.2.0 >= 5.4
OK:    Grep      3.11   >= 2.5.1a
OK:    Gzip      1.13   >= 1.3.12
OK:    M4        1.4.19 >= 1.4.10
OK:    Make      4.4.1  >= 4.0
OK:    Patch     2.8    >= 2.5.4
OK:    Perl      5.40.1 >= 5.8.8
OK:    Python    3.13.5 >= 3.4
OK:    Sed       4.9    >= 4.1.5
OK:    Tar       1.35   >= 1.22
OK:    Texinfo   7.1.1  >= 5.0
OK:    Xz        5.8.1  >= 5.0.0
OK:    Linux Kernel 6.12.57 >= 5.4
OK:    Linux Kernel supports UNIX 98 PTY
Aliases:
OK:    awk  is GNU
OK:    yacc is Bison
OK:    sh   is Bash
Compiler check:
OK:    g++ works
OK: nproc reports 6 logical cores are available

---

## Disk Partitioning
- [Create Disks](#create-disks)
- [Formatting and Mounting Disks](#formatting-and-mounting-disks)

### Create Disks

So now we need to have ```three``` partitions:

- ```boot```
: This will be about 500MB.

- ```swap```
: This is extra ram for when we build the distro. Should be the same size as our RAM allocation for the environment, so i'll make this 4GB.
- ```/```
: This is the root. This will just be the remaining space

First you'll need to know what your disk is called. To find this we use ```fdisk```

![fdiskhelp](/imgs/fdiskhelp.png)


Now theres a syntax to fdisk that you'll want to get familiar with. 
To add a partition, type ```fdisk /dev/sda```
The series of commands is so:
- ```n```: Creates a new partition
- ```p```: Is a primary partition
- ```1```: Set partition number
- The next step assigns the first sector. This is the starting point of our partition and we can just press ```enter``` to select the default.
- The next part gives us the option to resize in a handy way. We dont beent to calculate the exact bytes needed to make 500MB forom sector 2048 (it's XXX, but that's besides the point). We just need to type ```+500M``` to make the partition 500MB total.

![createpart](/imgs/createpart.png)

You'll need to change the drive type of the ```swap``` to Linux swap / Solaris aswell.
- ```t```: Change drive type
- Select which drive is the swap. For me its ```2```.
- If unsure, press ```L``` to list all options. The one you want is ```82```, so type that.

Afterwards when you press ```p``` to list all of partitions, you should see this

Now press ```w``` to write all of the partitions and ```q``` to exit.

![partitions](/imgs/partitions.png)

### Formatting and Mounting Disks

Once created, it's a good idea to format the drives and initialise the swap.

We can use ```mkfs.ext4``` to achieve the disk formatting and ```mkswap``` to enable swap.


In [ ]:

mkfs.ext4 /dev/sda1 # This will format our `boot` partition

mkswap /dev/sda2 # This creates our swap

mkfs.ext4 /dev/sda3 # This formats our root partition


We will need to export the variable for LFS

In [ ]:

export LFS=/mnt/lfs # Use echo $LFS to check, should say '/mnt'

Also set the file mode creation mask (umask) to ```021```. This controls default permissions for new files/directories by removing bits from the system's base permissions (666 for files, 777 for directories)


In [ ]:
umask 021

Now everything needs to be mounted 

In [ ]:

mkdir -pv $LFS                      # Create our root ('/') directory
mount -v -t ext4 /dev/sda3 $LFS     # Mount the / partition

chown user:user $LFS                # Give LFS user permissions over these directories
chmod 755 $LFS                      # Give the directory permissions

swapon /dev/sda2                    # Enables the swap 

This should just work, but we'll need to check if the swap partition is active.
We can use ```free -h``` to find this out.

![swap](/imgs/swap.png)

# Setting the LFS Environment

- [Create Directory Layout](#create-directory-layout)
- [Adding the LFS User](#adding-the-lfs-user)
- [Logging In](#logging-in)
- [Activate Environment](#activate-environment)
- [Configure Student-Specific Settings](#configure-student-specific-settings)

Great, you've made it this far. Now we get to start working on the actual task.

We will need to export the variable for LFS



We will need this because we will be referencing it hundres of times and its just easier this way

Now we need to create the source directory

In [ ]:
mkdir -v $LFS/sources # Creates /mnt/lfs/sources

chmod -v a+wt $LFS/sources
    # a+w = all users can write (needed because we'll build as different users)
    # t = "sticky bit" - users can only delete their own files (security measure)

## Pull Packages

There is a list of packages given in the LFS book. you may need to double check the sources because some servers may be down. I have included the ```wget-list.txt``` to add with this command

In [ ]:
wget --input-file=wget-list.txt --continue --directory-prefix=$LFS/sources

this should download all of the packages to ```LFS/sources```. If any are missing, you'll need to get them manually. After this has finished, changed the permissions to include the user

In [ ]:
sudo chown -R user:user $LFS/sources

---
### Create Directory Layout

We need to create a proper layout in our newly created space. 

The directories we need:
- ```/etc``` = system configuration files will go here
- ```/var``` = variable data (logs, caches, etc.)
- ```/usr/bin``` = user programs and commands
- ```/usr/lib``` = shared libraries
- ```/usr/sbin``` = system administration binaries

symbolic links:
Modern Linux systems are moving toward a "merged /usr" layout where:

- /bin → /usr/bin (symlink)
- /lib → /usr/lib (symlink)
- /sbin → /usr/sbin (symlink)

Old programs expect /bin/bash, but the actual file lives in /usr/bin/bash. The symlinks make both paths work.

The lib64 directory:
- Only created on 64-bit systems (x86_64)
- Some programs specifically look for 64-bit libraries in /lib64
- This ensures compatibility with software that has hardcoded paths

Here's a script to do this

In [ ]:
mkdir -pv $LFS/{etc,var} $LFS/usr/{bin,lib,sbin}

for i in bin lib sbin; do
  ln -sv usr/$i $LFS/$i
done

case $(uname -m) in
  x86_64) mkdir -pv $LFS/lib64 ;;
esac

And make a tools folder

In [ ]:
mkdir -pv $LFS/tools

---
### Adding the LFS User

Why create a user if we are root and can just do it from here. There are a few reasons

- ```Isolation```: The LFS user has no special privileges and can't accidentally damage the host system
- ```Clean environment```: The LFS user starts with a minimal environment - no weird environment variables from root that could break the build

- ```Prevents mistakes```: If you run a command like rm -rf /bin while logged in as root, you could destroy the host. As the lfs user, you can only affect the /mnt directory you own.

- ```Follows LFS best practices```: The book is designed around this workflow

Here's a script to help with this:

In [ ]:
groupadd lfs
useradd -s /bin/bash -g lfs -m -k /dev/null lfs
passwd lfs

chown -v lfs $LFS/{usr{,/*},lib,var,etc,bin,sbin,sources,tools}
case $(uname -m) in
  x86_64) chown -v lfs $LFS/lib64 ;;
esac

Breaking down the ```useradd``` command:

- ```-s /bin/bash``` = set bash as the shell
- ```-g lfs``` = primary group is "lfs"
- ```-m``` = create a home directory
- ```-k /dev/null``` = don't copy skeleton files (we want a truly clean environment)

The chown command:

- Gives the LFS user ownership of all the directories they'll need to write to during the build
- This way the LFS user can create files without needing root privileges

---
### Logging in

Now we get to switch to the LFS user

In [ ]:
su - lfs

In this command, the ```-``` creates a "login shell" which gives you a fresh environment. Without it, you'd inherit root's environment variables which could cause build issues.

![login](/imgs/login.png)

Now we need to set the LFS user's build environment

In [ ]:
export LFS=/mnt/lfs
cd $LFS/sources

---
### Activate Environment

Doing this is super simple, we've probably all done this before

In [ ]:
cat > ~/.bash_profile << "EOF"
exec env -i HOME=$HOME TERM=$TERM PS1='\u:\w\$ ' /bin/bash
EOF

cat > ~/.bashrc << "EOF"
set +h
umask 022
LFS=/mnt/lfs
LC_ALL=POSIX
LFS_TGT=$(uname -m)-lfs-linux-gnu
PATH=/usr/bin
if [ ! -L /bin ]; then PATH=/bin:$PATH; fi
PATH=$LFS/tools/bin:$PATH
CONFIG_SITE=$LFS/usr/share/config.site
export LFS LC_ALL LFS_TGT PATH CONFIG_SITE
EOF

export MAKEFLAGS=-j32 #allow 'make' to spawn up to 32 build jobs

cat >> ~/.bashrc << "EOF"
export MAKEFLAGS=-j$(nproc)
EOF

source ~/.bash_profile

This applies the changes to our current session and for all future sessions

Then you wil want to ```echo``` all of these to make sure they are correct

In [ ]:
echo $LFS        # Should show: /mnt/lfs
echo $LFS_TGT    # Should show: x86_64-lfs-linux-gnu (or similar)
echo $PATH       # Should start with: /mnt/tools/bin

### Configure Student-Specific Settings
We will need some values for this project. 

Why is this neccesary?
- When you build the kernel, you'll configure it to use:
    - Version string: `6.16.1-mgeiger`
    - This shows up in `uname -r` and identifies your custom kernel
- Bootloader Configuration
    - Your GRUB config will reference: ```/boot/vmlinuz-6.16.1-mgeiger```

So we can do this to add the variables to the bashrc:

In [ ]:
cat >> ~/.bashrc << "EOF"

# Student-specific configuration
STUDENT_LOGIN="mgeiger" # Use your's obviously
KERNEL_VERSION="6.16.1"
KERNEL_NAME="${KERNEL_VERSION}-${STUDENT_LOGIN}"
export STUDENT_LOGIN KERNEL_VERSION KERNEL_NAME
EOF

Then we just re-source from bashrc


In [ ]:
source ~/.bashrc


and then check with ```env```

![env](/imgs/export.png)

Later, we will need to set the ```hostname``` but the LFS user is not in the sudoers group (for good reason!!!)


# Building the Packages

- [binutils](#binutils)
- [GCC](#gcc)
- [Linux API Headers](#linux-api-headers)
- [Libstdc++](#libstdc)
- [M4](#m4)
- [Ncurses](#ncurses)
- [Bash](#bash)
- [Coreutils](#coreutils)
- [Diffutils](#diffutils)
- [File](#file)
- [Findutils](#findutils)
- [Gawk](#gawk)
- [Grep](#grep)
- [Gzip](#gzip)
- [Make](#make)
- [Patch](#patch)
- [Sed](#sed)
- [Tar](#tar)
- [Xz](#xz)
- [Binutils (2nd Pass)](#binutils-2nd-pass)
- [GCC (2nd Pass)](#gcc-2nd-pass)
<br>

---

## binutils

Lets start with out old friend ```binutils```. Binutils is a crucial collection of command-line tools from the GNU Project, essential for creating, manipulating, and inspecting binary files (executables, object files, libraries) in software development, featuring core tools like the as assembler, ld linker, strip (removes symbols), objdump (displays info), nm (lists symbols), and objcopy (copies/translates files).

This configures Binutils to build a cross-assembler and linker.<br>

In [ ]:
tar -xf binutils-2.45.1.tar.xz # 'Unzip' the tarball
cd binutils-2.45.1            #  Move into the directory
mkdir -v build               #  Create buld directory
cd build                     #  Move into the build directory
../configure --prefix=$LFS/tools \
             --with-sysroot=$LFS \
             --target=$LFS_TGT   \
             --disable-nls       \
             --enable-gprofng=no \
             --disable-werror    \
             --enable-new-dtags  \
             --enable-default-hash-style=gnu
make
make install
cd ../.. 
rm -rf binutils-2.45.1

---

## GCC

GCC most commonly refers to the GNU Compiler Collection, a vital, free, open-source suite of compilers for languages like C, C++, Fortran, and Ada, essential for Linux/Unix development.

We will need this to compile our code

In [ ]:
tar -xf gcc-15.2.0.tar.gz 
cd gcc-15.2.0

# GCC needs GMP, MPFR, and MPC - extract them into the gcc directory

tar -xf ../mpfr-4.2.2.tar.xz
mv -v mpfr-4.2.2 mpfr
tar -xf ../gmp-6.3.0.tar.xz
mv -v gmp-6.3.0 gmp
tar -xf ../mpc-1.3.1.tar.gz
mv -v mpc-1.3.1 mpc

# Why extract these inside GCC?
# - GCC needs these math libraries to compile
# - Putting them here makes GCC build them internally

# Now we have to configure GCC for cross compilation
case $(uname -m) in
  x86_64)
    sed -e '/m64=/s/lib64/lib/' \
        -i.orig gcc/config/i386/t-linux64
 ;;
esac

# Then the build directory
mkdir -v build
cd build

../configure                  \
    --target=$LFS_TGT         \
    --prefix=$LFS/tools       \
    --with-glibc-version=2.42 \
    --with-sysroot=$LFS       \
    --with-newlib             \
    --without-headers         \
    --enable-default-pie      \
    --enable-default-ssp      \
    --disable-nls             \
    --disable-shared          \
    --disable-multilib        \
    --disable-threads         \
    --disable-libatomic       \
    --disable-libgomp         \
    --disable-libquadmath     \
    --disable-libssp          \
    --disable-libvtv          \
    --disable-libstdcxx       \
    --enable-languages=c,c++

make

make install

# create a compatibility symlink
cd ..
cat gcc/limitx.h gcc/glimits.h gcc/limity.h > \
  `dirname $($LFS_TGT-gcc -print-libgcc-file-name)`/include/limits.h

# and again, we clean up
cd ..
rm -rf gcc-15-2-0


---

## Linux API Headers

The next thing we need to do is install the API headers. The Linux kernel needs to expose an Application Programming Interface (API) for the system's C library (Glibc in LFS) to use. This is done by way of sanitizing various C header files that are shipped in the Linux kernel source tarball.


In [ ]:
tar -xf linux-6.17.9.tar.xz # Extract the next tarball
cd linux-6.17.9             # Move into the directory

make mrproper # Clean the source tree

# Install the headers
make headers
find usr/include -type f ! -name '*.h' -delete
cp -rv usr/include $LFS/usr

# and again, we clean up
cd ..
rm -rf linux-6.17.9

---

## Glibc

This ones a big one...

Glibc (GNU C Library) is the essential C standard library for GNU/Linux systems, providing fundamental functions (like printf, malloc, file I/O, networking) that act as a bridge between applications and the Linux kernel, ensuring software portability and standard compliance (POSIX, ISO C) across distributions, making it a core component for almost all Linux programs

What does this ```make``` do?
- Building the C library (the foundation of your LFS system)
- Creating essential system calls and functions
- Building locale support, time functions, threading, etc.

In [ ]:
tar -xf glibc-2.42.tar.xz # Extract glibc

cd glibc-2.42   # Move into the directory

# This patch makes glibc compliant with the Filesystem Hierarchy Standard (FHS) and ensures libraries go in the correct locations
patch -Np1 -i ../glibc-2.42-fhs-1.patch # Patch the files

# Now we create our build directory
mkdir -v build
cd build

# Set the config parameters
echo "rootsbindir=/usr/sbin" > configparms 

# Run the configure command
../configure                             \
      --prefix=/usr                      \
      --host=$LFS_TGT                    \
      --build=$(../scripts/config.guess) \
      --disable-nscd                     \
      libc_cv_slibdir=/usr/lib           \
      --enable-kernel=5.4

# When finished simply run 
make # This may take a while (~10 mins)

# Install to $LFS
make DESTDIR=$LFS install 

# Fix hardcoded path in ldd
sed '/RTLDLIST=/s@/usr@@g' -i $LFS/usr/bin/ldd 

# Then we'll do a critical sanity check to verify your toolchain works:
echo 'int main(){}' | $LFS_TGT-gcc -x c - -v -Wl,--verbose &> dummy.log
readelf -l a.out | grep ': /lib'
# Should output: [Requesting program interpreter: /lib64/ld-linux-x86-64.so.2]

# Now make sure that we're set up to use the correct start files
grep -E -o "$LFS/lib.*/S?crt[1in].*succeeded" dummy.log


# Verify that the compiler is searching for the correct header files
grep -B3 "^ $LFS/usr/include" dummy.log


# Next, verify that the new linker is being used with the correct search paths
grep 'SEARCH.*/usr/lib' dummy.log |sed 's|; |\n|g'

# Check that we are using the correct libc
grep "/lib.*/libc.so.6 " dummy.log

#Make sure GCC is using the correct dynamic linker:
grep found dummy.log

# Cleanup
rm -v a.out dummy.log

cd ../..
rm -rf glibc-2.42

---

## Libstdc++

Libstdc++ is crucial for C++ programs compiled with the GNU Compiler Collection (GCC) (g++) on Linux and other systems, providing essential components like containers, algorithms, and I/O streams

For libstdc++, we need a clean gcc again

In [ ]:
tar -xf gcc-15.2.0.tar.gz
cd gcc-15.2.0

# then build directories for libstdc++
mkdir -v build
cd build

# Configure libstdc++
../libstdc++-v3/configure      \
    --host=$LFS_TGT            \
    --build=$(../config.guess) \
    --prefix=/usr              \
    --disable-multilib         \
    --disable-nls              \
    --disable-libstdcxx-pch    \
    --with-gxx-include-dir=/tools/$LFS_TGT/include/c++/15.2.0

# Then make
make

# We install with
make DESTDIR=$LFS install

# After that we remove the libtool archive files
rm -v $LFS/usr/lib/lib{stdc++{,exp,fs},supc++}.la

# Don't forget to clean up after yourself
cd ../..
rm -rf gcc-15.2.0

---

## M4

M4 is a powerful, general-purpose macro processor (a text-replacement tool) available on Unix-like systems, used to expand macros in source code (like C, Autoconf, Bison) before compilation

In [ ]:
tar -xf m4-1.4.20.tar.xz 
cd m4-1.4.20

# Configure m4
./configure --prefix=/usr   \
            --host=$LFS_TGT \
            --build=$(build-aux/config.guess)

# Then make
make

# We install with
make DESTDIR=$LFS install

# Don't forget to clean up after yourself
cd ..
rm -rf m4-1.4.20

---

##  Ncurses

Ncurses is a programming library for creating textual user interfaces (TUIs) that work across a wide variety of terminals

In [ ]:
tar -xf ncurses-6.5-20250809.tgz 
cd ncurses-6.5-20250809

mkdir -v build
pushd build
  ../configure --prefix=$LFS/tools AWK=gawk
  make -C include
  make -C progs tic
  install progs/tic $LFS/tools/bin
popd

# Configure m4
./configure --prefix=/usr                \
            --host=$LFS_TGT              \
            --build=$(./config.guess)    \
            --mandir=/usr/share/man      \
            --with-manpage-format=normal \
            --with-shared                \
            --without-normal             \
            --with-cxx-shared            \
            --without-debug              \
            --without-ada                \
            --disable-stripping          \
            AWK=gawk

# Then make
make

# We install with
make DESTDIR=$LFS install
ln -sv libncursesw.so $LFS/usr/lib/libncurses.so
sed -e 's/^#if.*XOPEN.*$/#if 1/' \
    -i $LFS/usr/include/curses.h

# Don't forget to clean up after yourself
cd ..
rm -rf ncurses-6.5-20250809

---

## Bash

Bash, which stands for "Bourne Again Shell," is
a command-line interpreter and scripting language for Unix-like operating systems like Linux and macOS. We are defintley going to need this.

In [ ]:
tar -xf bash-5.3.tar.gz
cd bash-5.3

# Configure bash
./configure --prefix=/usr                      \
            --build=$(sh support/config.guess) \
            --host=$LFS_TGT                    \
            --without-bash-malloc

# Then make
make

# We install with
make DESTDIR=$LFS install

# Make a link for the programs that use sh for a shell
ln -sv bash $LFS/bin/sh

# Don't forget to clean up after yourself
cd ..
rm -rf bash-5.3

---
## Coreutils

Coreutils
(GNU Core Utilities) are the essential collection of basic command-line programs for Unix-like operating systems (like Linux and macOS), providing fundamental file, text, and shell manipulation tools such as ls (list files), cp (copy), mv (move), rm (remove), and cat (concatenate), forming the bedrock for system administration and scripting, ensuring consistent functionality across diverse systems

In [ ]:
tar -xf coreutils-9.9.tar.xz 
cd coreutils-9.9

# Configure bash
./configure --prefix=/usr                     \
            --host=$LFS_TGT                   \
            --build=$(build-aux/config.guess) \
            --enable-install-program=hostname \
            --enable-no-install-program=kill,uptime

# Then make
make

# We install with
make DESTDIR=$LFS install

# Move programs to their final expected locations
mv -v $LFS/usr/bin/chroot              $LFS/usr/sbin
mkdir -pv $LFS/usr/share/man/man8
mv -v $LFS/usr/share/man/man1/chroot.1 $LFS/usr/share/man/man8/chroot.8
sed -i 's/"1"/"8"/'                    $LFS/usr/share/man/man8/chroot.8

# Don't forget to clean up after yourself
cd $LFS/sources
rm -rf coreutils-9.9

---
## Diffutils

Diffutils is a core GNU software package providing command-line tools like
diff, cmp, and diff3 for finding and displaying differences between files or directories, essential for version control and tracking changes in text-based data. It helps users see what's added, removed, or changed, generating output in various formats (like unified diffs) for patching or side-by-side viewing, crucial for developers and system administrators

In [ ]:
tar -xf diffutils-3.12.tar.xz
cd diffutils-3.12

# Configure diffutils
./configure --prefix=/usr   \
            --host=$LFS_TGT \
            gl_cv_func_strcasecmp_works=y \
            --build=$(./build-aux/config.guess)

# Then make
make

# We install with
make DESTDIR=$LFS install

# Don't forget to clean up after yourself
cd $LFS/sources
rm -rf diffutils-3.12

---
## File
The ```file``` command in Linux is a vital utility for determining the type of a file. It identifies file types by examining their content rather than their file extensions, making it an indispensable tool for users who work with various file formats.

In [ ]:
tar -xf file-5.46.tar.gz 
cd file-5.46

# Build step
mkdir -v build
pushd build
  ../configure --disable-bzlib      \
               --disable-libseccomp \
               --disable-xzlib      \
               --disable-zlib
  make
popd

# Prepare for compilation
./configure --prefix=/usr --host=$LFS_TGT --build=$(./config.guess)

# Then make
make FILE_COMPILE=$(pwd)/build/src/file

# We install with
make DESTDIR=$LFS install

# Remove the libtool archive file because it is harmful for cross compilation
rm -v $LFS/usr/lib/libmagic.la

# Don't forget to clean up after yourself
cd ../..
rm -rf file-5.46

---
## Findutils

Findutils is a core GNU software package providing essential command-line utilities for finding files and managing directories in Linux/Unix systems, primarily featuring the powerful find command (searches file hierarchies) and xargs (builds commands from input), along with locate and updatedb for fast, database-driven searches, offering versatile ways to locate and act on files

In [ ]:
tar -xf findutils-4.10.0.tar.xz 
cd findutils-4.10.0

# Configure findutils
./configure --prefix=/usr                   \
            --localstatedir=/var/lib/locate \
            --host=$LFS_TGT                 \
            --build=$(build-aux/config.guess)

# Then make
make

# We install with
make DESTDIR=$LFS install

# Don't forget to clean up after yourself
cd $LFS/sources
rm -rf findutils-4.10.0

---
## Gawk

The gawk command in Linux is a pattern scanning and processing language. No compilation is required, and variables can be used along with numeric functions, string functions, and logical operators

In [ ]:
tar -xf gawk-5.3.2.tar.xz 
cd gawk-5.3.2

# First, ensure some unneeded files are not installed
sed -i 's/extras//' Makefile.in

# Configure gawk
./configure --prefix=/usr   \
            --host=$LFS_TGT \
            --build=$(build-aux/config.guess)

# Then make
make

# We install with
make DESTDIR=$LFS install

# Don't forget to clean up after yourself
cd $LFS/sources
rm -rf gawk-5.3.2

---
## Grep

 Grep is the short form of 'global search for the regular expression'. The grep command is a filter that is used to search for lines matching a specified pattern and print the matching lines to standard output.

In [ ]:
tar -xf grep-3.12.tar.xz 
cd grep-3.12

# Configure grep
./configure --prefix=/usr   \
            --host=$LFS_TGT \
            --build=$(./build-aux/config.guess)

# Then make
make

# We install with
make DESTDIR=$LFS install

# Don't forget to clean up after yourself
cd $LFS/sources
rm -rf grep-3.12

---
## Gzip

Gzip is a command-line utility and file format for lossless data compression, reducing file sizes for storage or faster transfer, creating .gz files by default, and often paired with tar for archiving directories into tar.gz (tarball) files, using the DEFLATE algorithm

In [ ]:
tar -xf gzip-1.14.tar.xz 
cd gzip-1.14

# Configure gzip
./configure --prefix=/usr --host=$LFS_TGT

# Then make
make

# We install with
make DESTDIR=$LFS install

# Don't forget to clean up after yourself
cd $LFS/sources
rm -rf gzip-1.14

---
## Make

The make command for Linux is a very useful utility in the automation of software development and performing tasks in a Linux environment. It simply reads a special file, which is called a Makefile and this file describes how one's program is compiled and linked with another file or another program action.

In [ ]:
tar -xf make-4.4.1.tar.gz
cd make-4.4.1

# Configure make
./configure --prefix=/usr   \
            --host=$LFS_TGT \
            --build=$(build-aux/config.guess)

# Then make
make

# We install with
make DESTDIR=$LFS install

# Don't forget to clean up after yourself
cd $LFS/sources
rm -rf make-4.4.1

---
## Patch

Patch is a shell command that updates text files according to instructions in a separate file, called a patch file

In [ ]:
tar -xf patch-2.8.tar.xz
cd patch-2.8

# Configure patch
./configure --prefix=/usr   \
            --host=$LFS_TGT \
            --build=$(build-aux/config.guess)

# Then make
make

# We install with
make DESTDIR=$LFS install

# Don't forget to clean up after yourself
cd $LFS/sources
rm -rf patch-2.8

---
## Sed

The sed (Stream Editor) command in Linux is a powerful text processing tool used to perform basic text transformations on an input stream (a file or input from a pipeline). It allows you to search, replace, delete, and insert text, making it highly useful for automating text manipulation tasks.

In [ ]:
tar -xf sed-4.9.tar.xz 
cd sed-4.9

# Configure sed
./configure --prefix=/usr   \
            --host=$LFS_TGT \
            --build=$(./build-aux/config.guess)

# Then make
make

# We install with
make DESTDIR=$LFS install

# Don't forget to clean up after yourself
cd ..
rm -rf sed-4.9

---
## Tar

tar (Tape Archive) in Linux is a command-line utility that bundles multiple files and directories into a single archive file (.tar), making them easier to back up, transfer, and manage, while preserving file permissions and structure; it can also work with compression tools like gzip or bzip2 to create smaller, compressed archives (e.g., .tar.gz, .tar.bz2).

In [ ]:
tar -xf tar-1.35.tar.xz
cd tar-1.35

# Configure tar
./configure --prefix=/usr   \
            --host=$LFS_TGT \
            --build=$(build-aux/config.guess)

# Then make
make

# We install with
make DESTDIR=$LFS install

# Don't forget to clean up after yourself
cd ..
rm -rf tar-1.35

---
## Xz

XZ in Linux refers to the xz data compression tool and its .xz file format, known for high compression ratios using the LZMA2 algorithm, making it essential for shrinking Linux kernels, software packages, backups, and embedded system firmware, though it gained recent notoriety for a security backdoor found within its utilities.

In [ ]:
tar -xf xz-5.8.1.tar.xz 
cd xz-5.8.1

# Configure xz
./configure --prefix=/usr                     \
            --host=$LFS_TGT                   \
            --build=$(build-aux/config.guess) \
            --disable-static                  \
            --docdir=/usr/share/doc/xz-5.8.1

# Then make
make

# We install with
make DESTDIR=$LFS install

#  Remove the libtool archive file because it is harmful for cross compilation
rm -v $LFS/usr/lib/liblzma.la

# Don't forget to clean up after yourself
cd $LFS/sources
rm -rf xz-5.8.1

---
## Binutils (2nd Pass)

Why a second pass?

We perform two passes on Binutils (and other core tools) to create a self-contained, host-independent toolchain, first building temporary versions with the host's compiler (Pass 1) and then rebuilding them with the new, nascent toolchain (Pass 2) to strip host dependencies, ensuring the final system is built purely from LFS components for maximum portability and isolation

In [ ]:
tar -xf binutils-2.45.1.tar.xz 
cd binutils-2.45.1

# Binutils building system relies on an shipped libtool copy to link against internal static libraries, but the libiberty and zlib copies shipped in the package do not use libtool. This inconsistency may cause produced binaries mistakenly linked against libraries from the host distro. Work around this issue:
sed '6031s/$add_dir//' -i ltmain.sh

#Create a separate build directory again: 
mkdir -v build
cd build

# Prepare Binutils for compilation:
../configure                   \
    --prefix=/usr              \
    --build=$(../config.guess) \
    --host=$LFS_TGT            \
    --disable-nls              \
    --enable-shared            \
    --enable-gprofng=no        \
    --disable-werror           \
    --enable-64-bit-bfd        \
    --enable-new-dtags         \
    --enable-default-hash-style=gnu

# Then make
make

# We install with
make DESTDIR=$LFS install

#  Remove the libtool archive file because it is harmful for cross compilation
rm -v $LFS/usr/lib/lib{bfd,ctf,ctf-nobfd,opcodes,sframe}.{a,la}

# Don't forget to clean up after yourself
cd $LFS/sources
rm -rf binutils-2.45.1

---
## GCC (2nd Pass)

The same is true for our second pass of GCC. We want an isolated system that does not rely on the host machine

In [ ]:
tar -xf gcc-15.2.0.tar.gz  
cd gcc-15.2.0

# As in the first build of GCC, the GMP, MPFR, and MPC packages are required. Unpack the tarballs and move them into the required directories:
tar -xf ../mpfr-4.2.2.tar.xz
mv -v mpfr-4.2.2 mpfr
tar -xf ../gmp-6.3.0.tar.xz
mv -v gmp-6.3.0 gmp
tar -xf ../mpc-1.3.1.tar.gz
mv -v mpc-1.3.1 mpc

# Change the default directory name for 64-bit libraries to “lib”: 
case $(uname -m) in
  x86_64)
    sed -e '/m64=/s/lib64/lib/' \
        -i.orig gcc/config/i386/t-linux64
  ;;
esac

# Override the build rules of the libgcc and libstdc++ headers to allow building these libraries with POSIX threads support: 
sed '/thread_header =/s/@.*@/gthr-posix.h/' \
    -i libgcc/Makefile.in libstdc++-v3/include/Makefile.in

# Create a separate build directory again
mkdir -v build
cd build

# Prepare GCC for compilation:
../configure                   \
    --build=$(../config.guess) \
    --host=$LFS_TGT            \
    --target=$LFS_TGT          \
    --prefix=/usr              \
    --with-build-sysroot=$LFS  \
    --enable-default-pie       \
    --enable-default-ssp       \
    --disable-nls              \
    --disable-multilib         \
    --disable-libatomic        \
    --disable-libgomp          \
    --disable-libquadmath      \
    --disable-libsanitizer     \
    --disable-libssp           \
    --disable-libvtv           \
    --enable-languages=c,c++   \
    LDFLAGS_FOR_TARGET=-L$PWD/$LFS_TGT/libgcc

# Then make
make

# We install with
make DESTDIR=$LFS install

# As a finishing touch, create a utility symlink. Many programs and scripts run cc instead of gcc, 
# which is used to keep programs generic and therefore usable on all kinds of UNIX systems where 
# the GNU C compiler is not always installed. Running cc leaves the system administrator free to 
# cleardecide which C compiler to install
ln -sv gcc $LFS/usr/bin/cc

# Don't forget to clean up after yourself
cd $LFS/sources
rm -rf gcc-15.2.0

<br>

---

---

# Chroot

This is a major milestone. We are now going to use the environment we just created.

First you'll have to ```exit``` out of the LFS user and run this command to
change ownership of everything to root,

In [ ]:
sudo -i # Enter sudo

chown --from lfs -R root:root $LFS/{usr,var,etc,tools}
case $(uname -m) in
  x86_64) chown --from lfs -R root:root $LFS/lib64 ;;
esac

Then we need to prepare the kernel file system (KFS)

In [ ]:
# Create necessary directories
mkdir -pv $LFS/{dev,proc,sys,run}

# Create initial device nodes
mount -v --bind /dev $LFS/dev

mount -vt devpts devpts -o gid=5,mode=0620 $LFS/dev/pts
mount -vt proc proc $LFS/proc
mount -vt sysfs sysfs $LFS/sys
mount -vt tmpfs tmpfs $LFS/run

# If /dev/shm exists
if [ -h $LFS/dev/shm ]; then
  install -v -d -m 1777 $LFS$(realpath /dev/shm)
else
  mount -vt tmpfs -o nosuid,nodev tmpfs $LFS/dev/shm
fi

# Create symlink from /lib64 to the actual dynamic linker in /usr/lib
ln -sv ../usr/lib/ld-linux-x86-64.so.2 $LFS/lib64/ld-linux-x86-64.so.2

# Also ensure libc.so.6 is accessible
ln -sv ../usr/lib/libc.so.6 $LFS/lib64/libc.so.6

# Verify the symlinks
ls -la $LFS/lib64/

Drumroll please......

In [ ]:
# Enter Chroot!
chroot "$LFS" /usr/bin/env -i   \
    HOME=/root                  \
    TERM="$TERM"                \
    PS1='(lfs chroot) \u:\w\$ ' \
    PATH=/usr/bin:/usr/sbin     \
    MAKEFLAGS="-j$(nproc)"      \
    TESTSUITEFLAGS="-j$(nproc)" \
    /bin/bash --login

### Congratulations, you are now in the new LFS system!!!!
your prompt should now look like

In [ ]:
(lfs chroot) I have no name!:/#

## Kicked Out!

If you need to get back into the system to add more packages or do upgrades, you'll need to remount the drives and re-enter chroot

In [ ]:
# 1. Mount the LFS partition
export LFS=/mnt/lfs
sudo mkdir -pv $LFS
sudo mount -v -t ext4 /dev/sda3 $LFS

# 2. Activate swap (sda2)
sudo swapon /dev/sda2

# 3. Mount the virtual kernel file systems
sudo mount -v --bind /dev $LFS/dev
sudo mount -v --bind /dev/pts $LFS/dev/pts
sudo mount -vt proc proc $LFS/proc
sudo mount -vt sysfs sysfs $LFS/sys
sudo mount -vt tmpfs tmpfs $LFS/run

# 4. Enter the chroot environment
sudo chroot "$LFS" /usr/bin/env -i   \
    HOME=/root                  \
    TERM="$TERM"                \
    PS1='(lfs chroot) \u:\w\$ ' \
    PATH=/usr/bin:/usr/sbin     \
    /bin/bash --login

## Fix SSH

To fix the known hosts issue, just run this command on the thost machine

In [ ]:
ssh-keygen -f "/root/.ssh/known_hosts" -R "[localhost]:2222"

# Set Up Your Environment 

First off you'll need to to create some essential directories and file

In [ ]:
# Create the FHS directory structure
mkdir -pv /{boot,home,mnt,opt,srv}
mkdir -pv /etc/{opt,sysconfig}
mkdir -pv /lib/firmware
mkdir -pv /media/{floppy,cdrom}
mkdir -pv /usr/{,local/}{include,src}
mkdir -pv /usr/lib/locale
mkdir -pv /usr/local/{bin,lib,sbin}
mkdir -pv /usr/{,local/}share/{color,dict,doc,info,locale,man}
mkdir -pv /usr/{,local/}share/{misc,terminfo,zoneinfo}
mkdir -pv /usr/{,local/}share/man/man{1..8}
mkdir -pv /var/{cache,local,log,mail,opt,spool}
mkdir -pv /var/lib/{color,misc,locate}

# Create compatibility symlink
ln -sfv /run /var/run
ln -sfv /run/lock /var/lock

# Set permissions
install -dv -m 0750 /root
install -dv -m 1777 /tmp /var/tmp

## Create Essential Files and Symlinks

In [ ]:
ln -sv /proc/self/mounts /etc/mtab

# Create /etc/hosts
cat > /etc/hosts << EOF
127.0.0.1  localhost $(hostname)
::1        localhost
EOF

# Create /etc/passwd
cat > /etc/passwd << "EOF"
root:x:0:0:root:/root:/bin/bash
bin:x:1:1:bin:/dev/null:/usr/bin/false
daemon:x:6:6:Daemon User:/dev/null:/usr/bin/false
messagebus:x:18:18:D-Bus Message Daemon User:/run/dbus:/usr/bin/false
uuidd:x:80:80:UUID Generation Daemon User:/dev/null:/usr/bin/false
nobody:x:65534:65534:Unprivileged User:/dev/null:/usr/bin/false
EOF

# Create /etc/group
cat > /etc/group << "EOF"
root:x:0:
bin:x:1:daemon
sys:x:2:
kmem:x:3:
tape:x:4:
tty:x:5:
daemon:x:6:
floppy:x:7:
disk:x:8:
lp:x:9:
dialout:x:10:
audio:x:11:
video:x:12:
utmp:x:13:
cdrom:x:15:
adm:x:16:
messagebus:x:18:
input:x:24:
mail:x:34:
kvm:x:61:
uuidd:x:80:
wheel:x:97:
users:x:999:
nogroup:x:65534:
EOF

# Add our own user
echo "mgeiger:x:101:101::/home/mgeiger:/bin/bash" >> /etc/passwd
echo "mgeiger:x:101:" >> /etc/group
install -o mgeiger -d /home/mgeiger

# Then login
exec /usr/bin/bash --login

# New prompt
#(lfs chroot) root:/# 

# Initialize log files
touch /var/log/{btmp,lastlog,faillog,wtmp}
chgrp -v utmp /var/log/lastlog
chmod -v 664  /var/log/lastlog
chmod -v 600  /var/log/btmp

<br>

## From here (/), we can enter ```/sources``` and that is our old ```/mnt/lfs/sources```

In [ ]:
cd /sources
pwd # /sources
ls
# Python-3.14.0.tar.xz               libcap-2.77.tar.xz
# XML-Parser-2.47.tar.gz             libffi-3.5.2.tar.gz
# acl-2.3.2.tar.xz                   libtool-2.5.4.tar.xz
# attr-2.5.2.tar.gz                  libxcrypt-4.5.2.tar.xz
# autoconf-2.72.tar.xz               linux-6.17.9
# .......

---

---

# Next Packages to Install

This part is gonna take A LOOOOT of time. Grab a coffee or whatever, sit back and start building everything you need from source. I wish you luck!

## Gettext

In [ ]:
tar -xf gettext-0.26.tar.xz 
cd gettext-0.26

# Configure gettext
./configure --disable-shared

# Then make
make

# Install the msgfmt, msgmerge, and xgettext programs:
cp -v gettext-tools/src/{msgfmt,msgmerge,xgettext} /usr/bin

# Don't forget to clean up after yourself
cd ..
rm -rf gettext-0.26

## Bison

In [ ]:
tar -xf bison-3.8.2.tar.xz 
cd bison-3.8.2 

# Configure bison
./configure --prefix=/usr \
            --docdir=/usr/share/doc/bison-3.8.2

# Then make
make

# Install 
make install

# Don't forget to clean up after yourself
cd ..
rm -rf bison-3.8.2

## Perl

In [ ]:
tar -xf perl-5.42.0.tar.gz 
cd perl-5.42.0

# Configure perl
sh Configure -des                                         \
             -D prefix=/usr                               \
             -D vendorprefix=/usr                         \
             -D useshrplib                                \
             -D privlib=/usr/lib/perl5/5.42/core_perl     \
             -D archlib=/usr/lib/perl5/5.42/core_perl     \
             -D sitelib=/usr/lib/perl5/5.42/site_perl     \
             -D sitearch=/usr/lib/perl5/5.42/site_perl    \
             -D vendorlib=/usr/lib/perl5/5.42/vendor_perl \
             -D vendorarch=/usr/lib/perl5/5.42/vendor_perl

# Then make
make

# Install 
make install

# Don't forget to clean up after yourself
cd ..
rm -rf perl-5.42.

## Python

In [ ]:
tar -xf Python-3.14.0.tar.xz  
cd Python-3.14.0

# Configure python
./configure --prefix=/usr       \
            --enable-shared     \
            --without-ensurepip \
            --without-static-libpython

# Then make
make

# Install 
make install

# Don't forget to clean up after yourself
cd ..
rm -rf Python-3.14.0

## Texinfo

In [ ]:
tar -xf texinfo-7.2.tar.xz
cd texinfo-7.2

# Configure texinfo
./configure --prefix=/usr

# Then make
make

# Install 
make install

# Don't forget to clean up after yourself
cd ..
rm -rf texinfo-7.2

## Util-linux

In [ ]:
tar -xf util-linux-2.41.2.tar.xz 
cd util-linux-2.41.2

#  The FHS recommends using the /var/lib/hwclock directory instead of the usual /etc directory as the location for the adjtime file. Create this directory with
mkdir -pv /var/lib/hwclock

# Configure texinfo
./configure --libdir=/usr/lib     \
            --runstatedir=/run    \
            --disable-chfn-chsh   \
            --disable-login       \
            --disable-nologin     \
            --disable-su          \
            --disable-setpriv     \
            --disable-runuser     \
            --disable-pylibmount  \
            --disable-static      \
            --disable-liblastlog2 \
            --without-python      \
            ADJTIME_PATH=/var/lib/hwclock/adjtime \
            --docdir=/usr/share/doc/util-linux-2.41.1

# Then make
make

# Install 
make install

# Don't forget to clean up after yourself
cd ..
rm -rf util-linux-2.41.2

# Cleaning up and Saving the Temporary System

In [ ]:
# =============================================================
# CLEANING UP THE TEMPORARY SYSTEM (Still inside chroot!)
# =============================================================

# First, remove the currently installed documentation files to prevent them 
# from ending up in the final system, and to save about 35 MB
rm -rf /usr/share/{info,man,doc}/*

# Second, on a modern Linux system, the libtool .la files are only useful 
# for libltdl. No libraries in LFS are loaded by libltdl, and it's known 
# that some .la files can cause BLFS package failures. Remove those files now
find /usr/{lib,libexec} -name \*.la -delete

# The current system size is now about 3 GB, however the /tools directory 
# is no longer needed. It uses about 1 GB of disk space. Delete it now: 
rm -rf /tools

# =============================================================
# ⚠️  OPTIONAL: EXIT ONLY TO CREATE A BACKUP ⚠️
# =============================================================
# If you want to create a backup tarball, exit now with 'exit'
# Otherwise, SKIP THE EXIT and continue to the next package!
#
# If you do exit, you MUST re-enter chroot before continuing:
#
#   # On host (as root):
#   export LFS=/mnt/lfs
#   mount -v --bind /dev $LFS/dev
#   mount -vt devpts devpts -o gid=5,mode=0620 $LFS/dev/pts
#   mount -vt proc proc $LFS/proc
#   mount -vt sysfs sysfs $LFS/sys
#   mount -vt tmpfs tmpfs $LFS/run
#
#   chroot "$LFS" /usr/bin/env -i   \
#       HOME=/root                  \
#       TERM="$TERM"                \
#       PS1='(lfs chroot) \u:\w\$ ' \
#       PATH=/usr/bin:/usr/sbin     \
#       MAKEFLAGS="-j$(nproc)"      \
#       /bin/bash --login
#
# =============================================================
# ⚠️  ALL PACKAGES FROM HERE ON ARE BUILT INSIDE CHROOT ⚠️
# =============================================================



# Next Set of Package

## Man Pages

In [ ]:
# Extract and enter
tar -xf man-pages-6.16.tar.xz
cd man-pages-6.16

# Remove two man pages for password hashing functions. 
# Libxcrypt will provide a better version of these man pages
rm -v man3/crypt*

# Install Man-pages by running
make -R GIT=false prefix=/usr install

# Clean up
cd ..
rm -rf man-pages-6.16

## Iana-Etc

In [ ]:
# Extract and enter
tar -xf iana-etc-20251120.tar.gz
cd iana-etc-20251120

# For this package, we only need to copy the files into place
cp services protocols /etc

# Clean up
cd ..
rm -rf iana-etc-20251120

## Glibc

In [ ]:
tar -xf glibc-2.42.tar.xz
cd glibc-2.42

# Some of the Glibc programs use the non-FHS compliant /var/db directory 
# to store their runtime data. Apply the following patch to make such 
# programs store their runtime data in the FHS-compliant locations: 
patch -Np1 -i ../glibc-2.42-fhs-1.patch

# Now fix an issue which may break Valgrind in BLFS
sed -e '/unistd.h/i #include <string.h>' \
    -e '/libc_rwlock_init/c\
  __libc_rwlock_define_initialized (, reset_lock);\
  memcpy (&lock, &reset_lock, sizeof (lock));' \
    -i stdlib/abort.c

# Build dirs
mkdir -v build
cd build

# Ensure that the ldconfig and sln utilities will be installed into /usr/sbin
echo "rootsbindir=/usr/sbin" > configparms

# Prepare Glibc for compilation: 
../configure --prefix=/usr                   \
             --disable-werror                \
             --disable-nscd                  \
             libc_cv_slibdir=/usr/lib        \
             --enable-stack-protector=strong \
             --enable-kernel=5.4

# Make
make

# Make check
make check

#Though it is a harmless message, the install stage of Glibc 
#will complain about the absence of /etc/ld.so.conf. Prevent this warning with:
touch /etc/ld.so.conf

#  Fix the Makefile to skip an outdated sanity check that fails with a modern Glibc configuration: 
sed '/test-installation/s@$(PERL)@echo not running@' -i ../Makefile

# Install
make install

#  Fix a hardcoded path to the executable loader in the ldd script: 
sed '/RTLDLIST=/s@/usr@@g' -i /usr/bin/ldd

# Install Locales
make localedata/install-locales

#  The /etc/nsswitch.conf file needs to be created because the Glibc defaults do not work well in a networked environment.

cat > /etc/nsswitch.conf << "EOF"
# Begin /etc/nsswitch.conf

passwd: files
group: files
shadow: files

hosts: files dns
networks: files

protocols: files
services: files
ethers: files
rpc: files

# End /etc/nsswitch.conf
EOF

# Adding time zone
tar -xf ../../tzdata2025b.tar.gz

ZONEINFO=/usr/share/zoneinfo
mkdir -pv $ZONEINFO/{posix,right}

for tz in etcetera southamerica northamerica europe africa antarctica  \
          asia australasia backward; do
    zic -L /dev/null   -d $ZONEINFO       ${tz}
    zic -L /dev/null   -d $ZONEINFO/posix ${tz}
    zic -L leapseconds -d $ZONEINFO/right ${tz}
done

cp -v zone.tab zone1970.tab iso3166.tab $ZONEINFO
zic -d $ZONEINFO -p America/New_York
unset ZONEINFO tz

tzselect

ln -sfv /usr/share/zoneinfo/<xxx> /etc/localtime

cd ../..
rm -rf glibc-2.42

## Zlib

In [ ]:
tar -xf zlib-1.3.1.tar.gz
cd zlib-1.3.1

./configure --prefix=/usr

make

make check

make install

cd ..
rm -rf zlib-1.3.1

## Bzip

In [ ]:
tar -xf bzip2-1.0.8.tar.gz
cd bzip2-1.0.8

patch -Np1 -i ../bzip2-1.0.8-install_docs-1.patch

sed -i 's@\(ln -s -f \)$(PREFIX)/bin/@\1@' Makefile

sed -i "s@(PREFIX)/man@(PREFIX)/share/man@g" Makefile

make -f Makefile-libbz2_so
make clean

make

make PREFIX=/usr install

cp -av libbz2.so.* /usr/lib
ln -sv libbz2.so.1.0.8 /usr/lib/libbz2.so

cp -v bzip2-shared /usr/bin/bzip2
for i in /usr/bin/{bzcat,bunzip2}; do
  ln -sfv bzip2 $i
done

rm -fv /usr/lib/libbz2.a

cd ..
rm -rf bzip-1.0.8

## Xz

In [ ]:
tar -xf xz-5.8.1.tar.xz
cd xz-5.8.1

./configure --prefix=/usr    \
            --disable-static \
            --docdir=/usr/share/doc/xz-5.8.1

make

make check

make install

cd ..
rm -rf xz-5.8.1

## Lz

In [ ]:
tar -xf lz4-1.10.0.tar.gz 
cd lz4-1.10.0

make BUILD_STATIC=no PREFIX=/usr

make -j1 check

make BUILD_STATIC=no PREFIX=/usr install

cd ..
rm -rf lz4-1.10.0

## Zstd

In [ ]:
tar -xf zstd-1.5.7.tar.gz
cd zstd-1.5.7

make prefix=/usr

make check

make prefix=/usr install

rm -v /usr/lib/libzstd.a

cd ..
rm -rf zstd-1.5.7

## File

In [ ]:
tar -xf file-5.46.tar.gz
cd file-5.46

./configure --prefix=/usr

make

make check

make install

cd ..
rm -rf file-5.46

## Readline

In [ ]:
tar -xf readline-8.3.tar.gz 
cd readline-8.3

sed -i '/MV.*old/d' Makefile.in
sed -i '/{OLDSUFF}/c:' support/shlib-install

sed -i 's/-Wl,-rpath,[^ ]*//' support/shobj-conf

./configure --prefix=/usr    \
            --disable-static \
            --with-curses    \
            --docdir=/usr/share/doc/readline-8.3

make SHLIB_LIBS="-lncursesw"

make install

install -v -m644 doc/*.{ps,pdf,html,dvi} /usr/share/doc/readline-8.3

cd ..
rm -rf readline-8.3

## M4

In [ ]:
tar -xf m4-1.4.20.tar.xz 
cd m4-1.4.20

./configure --prefix=/usr

make

make check

make install

cd ..
rm -rf m4-1.4.20

## Bc

In [ ]:
tar -xf bc-7.1.0.tar.xz 
cd bc-7.1.0

CC='gcc -std=c99' ./configure --prefix=/usr -G -O3 -r

make

make test

make install

cd ..
rm -rf bc-7.1.0

## Flex

In [ ]:
tar -xf flex-2.6.4.tar.gz 
cd flex-2.6.4

./configure --prefix=/usr \
            --docdir=/usr/share/doc/flex-2.6.4 \
            --disable-static

make

make check

make install

ln -sv flex   /usr/bin/lex
ln -sv flex.1 /usr/share/man/man1/lex.1

cd ..
rm -rf flex-2.6.4

## Tcl

In [ ]:
tar -xf tcl8.6.17-src.tar.gz
cd tcl8.6.17

SRCDIR=$(pwd)
cd unix
./configure --prefix=/usr           \
            --mandir=/usr/share/man \
            --disable-rpath

make

sed -e "s|$SRCDIR/unix|/usr/lib|" \
    -e "s|$SRCDIR|/usr/include|"  \
    -i tclConfig.sh

sed -e "s|$SRCDIR/unix/pkgs/tdbc1.1.10|/usr/lib/tdbc1.1.10|" \
    -e "s|$SRCDIR/pkgs/tdbc1.1.10/generic|/usr/include|"     \
    -e "s|$SRCDIR/pkgs/tdbc1.1.10/library|/usr/lib/tcl8.6|"  \
    -e "s|$SRCDIR/pkgs/tdbc1.1.10|/usr/include|"             \
    -i pkgs/tdbc1.1.10/tdbcConfig.sh

sed -e "s|$SRCDIR/unix/pkgs/itcl4.3.2|/usr/lib/itcl4.3.2|" \
    -e "s|$SRCDIR/pkgs/itcl4.3.2/generic|/usr/include|"    \
    -e "s|$SRCDIR/pkgs/itcl4.3.2|/usr/include|"            \
    -i pkgs/itcl4.3.2/itclConfig.sh

unset SRCDIR

make test

make install 
chmod 644 /usr/lib/libtclstub8.6.a

chmod -v u+w /usr/lib/libtcl8.6.so

make install-private-headers

ln -sfv tclsh8.6 /usr/bin/tclsh

mv /usr/share/man/man3/{Thread,Tcl_Thread}.3

cd ..
tar -xf ../tcl8.6.16-html.tar.gz --strip-components=1
cd tcl8.6.16
mkdir -v -p /usr/share/doc/tcl-8.6.16
cp -v -r  ./html/* /usr/share/doc/tcl-8.6.16

cd ..
rm -rf tcl8.6.17

## Expect

In [ ]:
tar -xf expect5.45.4.tar.gz 
cd expect5.45.4

python3 -c 'from pty import spawn; spawn(["echo", "ok"])'

patch -Np1 -i ../expect-5.45.4-gcc15-1.patch

./configure --prefix=/usr           \
            --with-tcl=/usr/lib     \
            --enable-shared         \
            --disable-rpath         \
            --mandir=/usr/share/man \
            --with-tclinclude=/usr/include

make

make test

make install
ln -svf expect5.45.4/libexpect5.45.4.so /usr/lib

cd ..
rm -rf expect5.45.4

## DejaGNU

In [ ]:
tar -xf dejagnu-1.6.3.tar.gz 
cd dejagnu-1.6.3

mkdir -v build
cd build

../configure --prefix=/usr
makeinfo --html --no-split -o doc/dejagnu.html ../doc/dejagnu.texi
makeinfo --plaintext       -o doc/dejagnu.txt  ../doc/dejagnu.texi

make check

make install
install -v -dm755  /usr/share/doc/dejagnu-1.6.3
install -v -m644   doc/dejagnu.{html,txt} /usr/share/doc/dejagnu-1.6.3

cd ../..
rm -rf dejagnu-1.6.3

## Pkgconf

In [ ]:
tar -xf pkgconf-2.5.1.tar.xz
cd pkgconf-2.5.1


./configure --prefix=/usr    \
            --disable-static \
            --docdir=/usr/share/doc/pkgconf-2.5.1

make

make install

ln -sv pkgconf   /usr/bin/pkg-config
ln -sv pkgconf.1 /usr/share/man/man1/pkg-config.1

cd ..
rm -rf pkgcong-2.5.1

## Binutils

In [ ]:
tar -xf binutils-2.45.1.tar.xz
cd binutils-2.45.1

mkdir -v build
cd build

../configure --prefix=/usr       \
             --sysconfdir=/etc   \
             --enable-ld=default \
             --enable-plugins    \
             --enable-shared     \
             --disable-werror    \
             --enable-64-bit-bfd \
             --enable-new-dtags  \
             --with-system-zlib  \
             --enable-default-hash-style=gnu

make tooldir=/usr

make -k check

grep '^FAIL:' $(find -name '*.log')

make tooldir=/usr install

rm -rfv /usr/lib/lib{bfd,ctf,ctf-nobfd,gprofng,opcodes,sframe}.a \
        /usr/share/doc/gprofng/

cd ../..
rm -rf binutils-2.45.1

## GMP

In [ ]:
tar -xf gmp-6.3.0.tar.xz
cd gmp-6.3.0

sed -i '/long long t1;/,+1s/()/(...)/' configure

./configure --prefix=/usr    \
            --enable-cxx     \
            --disable-static \
            --docdir=/usr/share/doc/gmp-6.3.0
make
make html

make check 2>&1 | tee gmp-check-log

awk '/# PASS:/{total+=$3} ; END{print total}' gmp-check-log

make install
make install-html

cd ..
rm -rf gmp-6.3.0

## MPFR

In [ ]:
tar -xf mpfr-4.2.2.tar.xz 
cd mpfr-4.2.2

./configure --prefix=/usr        \
            --disable-static     \
            --enable-thread-safe \
            --docdir=/usr/share/doc/mpfr-4.2.2

make
make html

make check

make install
make install-html

cd ..
rm -rf mpfr-4.2.2

## MPC

In [ ]:
tar -xf mpc-1.3.1.tar.gz
cd mpc-1.3.1

./configure --prefix=/usr    \
            --disable-static \
            --docdir=/usr/share/doc/mpc-1.3.1

make
make html

make check

make install
make install-html

cd ..
rm -rf mpc-1.3.1

## Attr

In [ ]:
tar -xf attr-2.5.2.tar.gz
cd attr-2.5.2

./configure --prefix=/usr     \
            --disable-static  \
            --sysconfdir=/etc \
            --docdir=/usr/share/doc/attr-2.5.2
make

make check

make install

cd ..
rm -rf attr-2.5.2

##  Acl

In [ ]:
tar -xf acl-2.3.2.tar.xz 
cd acl-2.3.2

./configure --prefix=/usr    \
            --disable-static \
            --docdir=/usr/share/doc/acl-2.3.2

make

make check

make install

cd ..
rm -rf acl-2.3.2

## Libcap

In [ ]:
tar -xf libcap-2.77.tar.xz
cd libcap-2.77

sed -i '/install -m.*STA/d' libcap/Makefile

make prefix=/usr lib=lib

make test

make prefix=/usr lib=lib install

cd ..
rm -rf libcap-2.77

## Libxcrypt

In [ ]:
tar -xf libxcrypt-4.5.2.tar.xz
cd libxcrypt-4.5.2

./configure --prefix=/usr                \
            --enable-hashes=strong,glibc \
            --enable-obsolete-api=no     \
            --disable-static             \
            --disable-failure-tokens

make

make check

make install

cd ..
rm -rf libxcrypt-4.5.2

## Shadow

In [ ]:
tar -xf shadow-4.18.0.tar.xz
cd shadow-4.18.0

sed -i 's/groups$(EXEEXT) //' src/Makefile.in
find man -name Makefile.in -exec sed -i 's/groups\.1 / /'   {} \;
find man -name Makefile.in -exec sed -i 's/getspnam\.3 / /' {} \;
find man -name Makefile.in -exec sed -i 's/passwd\.5 / /'   {} \;

sed -e 's:#ENCRYPT_METHOD DES:ENCRYPT_METHOD YESCRYPT:' \
    -e 's:/var/spool/mail:/var/mail:'                   \
    -e '/PATH=/{s@/sbin:@@;s@/bin:@@}'                  \
    -i etc/login.defs

touch /usr/bin/passwd
./configure --sysconfdir=/etc   \
            --disable-static    \
            --with-{b,yes}crypt \
            --without-libbsd    \
            --with-group-name-max-length=32

make

make exec_prefix=/usr install
make -C man install-man

pwconv

grpconv

mkdir -p /etc/default
useradd -D --gid 999

sed -i '/MAIL/s/yes/no/' /etc/default/useradd

passwd root

cd ..
rm -rf shadow-4.18.0

## Add new user to LFS

In [ ]:
useradd -m -G wheel -s /bin/bash mgeiger # or whatever
passwd mgeiger

In [ ]:

# Create sshd_config
cat > /etc/ssh/sshd_config << "SSHEOF"
# SchmitziOS SSH Server Configuration

# Port and Address
Port 22

# Host Keys
HostKey /etc/ssh/ssh_host_rsa_key
HostKey /etc/ssh/ssh_host_ecdsa_key
HostKey /etc/ssh/ssh_host_ed25519_key

# Authentication
PermitRootLogin yes
PubkeyAuthentication yes
PasswordAuthentication yes
PermitEmptyPasswords no
ChallengeResponseAuthentication no

# PAM
UsePAM no

# Allow client to pass locale environment variables
AcceptEnv LANG LC_*

# Override default of no subsystems
Subsystem	sftp	/usr/libexec/sftp-server

# Logging
SyslogFacility AUTH
LogLevel INFO
SSHEOF

# Generate SSH host keys
ssh-keygen -A


# CRITICAL: Set root password using chpasswd for reliability
echo 'root:password' | chpasswd

# Verify root password was set
if grep -q '^root:[^:]*[^:]:' /etc/shadow; then
    echo "✓ Root password set successfully"
else
    echo "✗ WARNING: Root password may not be set correctly!"
    echo "  Manual fallback: run 'passwd root' and set password"
fi


## SSH Server Configuration

Configure SSH to start automatically with secure defaults.

In [ ]:
cd /sources
mkdir -p openssh && cd openssh

# Download OpenSSH
curl -L -O http://ftp.openbsd.org/pub/OpenBSD/OpenSSH/portable/openssh-9.8p1.tar.gz || \
wget --no-check-certificate https://cdn.openbsd.org/pub/OpenBSD/OpenSSH/portable/openssh-9.8p1.tar.gz

tar -xzf openssh-9.8p1.tar.gz
cd openssh-9.8p1

# Configure OpenSSH
./configure --prefix=/usr \
            --sysconfdir=/etc/ssh \
            --with-privsep-path=/var/lib/sshd \
            --with-default-path=/usr/bin \
            --with-superuser-path=/usr/sbin:/usr/bin \
            --with-pid-dir=/run \
            --with-md5-passwords

# Compile
make

# Install
make install

# Create privilege separation directory
install -v -m700 -d /var/lib/sshd
chown -v root:sys /var/lib/sshd 2>/dev/null || chown -v root:root /var/lib/sshd

# Create SSH directories
mkdir -p /etc/ssh
mkdir -p /root/.ssh
chmod 700 /root/.ssh

## GCC

In [ ]:
tar -xf gcc-15.2.0
cd gcc-15.2.0

case $(uname -m) in
  x86_64)
    sed -e '/m64=/s/lib64/lib/' \
        -i.orig gcc/config/i386/t-linux64
  ;;
esac

mkdir -v build
cd build

../configure --prefix=/usr            \
             LD=ld                    \
             --enable-languages=c,c++ \
             --enable-default-pie     \
             --enable-default-ssp     \
             --enable-host-pie        \
             --disable-multilib       \
             --disable-bootstrap      \
             --disable-fixincludes    \
             --with-system-zlib
make

ulimit -s -H unlimited

sed -e '/cpython/d' -i ../gcc/testsuite/gcc.dg/plugin/plugin.exp

chown -R mgeiger .
su mgeiger -c "PATH=$PATH make -k check"

../contrib/test_summary

make install

chown -v -R root:root /usr/lib/gcc/$(gcc -dumpmachine)/15.2.0/include{,-fixed}

ln -svr /usr/bin/cpp /usr/lib

ln -sv gcc.1 /usr/share/man/man1/cc.1

ln -sfv ../../libexec/gcc/$(gcc -dumpmachine)/15.2.0/liblto_plugin.so \
        /usr/lib/bfd-plugins/

echo 'int main(){}' | cc -x c - -v -Wl,--verbose &> dummy.log
readelf -l a.out | grep ': /lib'

grep -E -o '/usr/lib.*/S?crt[1in].*succeeded' dummy.log

grep -B4 '^ /usr/include' dummy.log

grep 'SEARCH.*/usr/lib' dummy.log |sed 's|; |\n|g'

grep "/lib.*/libc.so.6 " dummy.log

grep found dummy.log

rm -v a.out dummy.log

mkdir -pv /usr/share/gdb/auto-load/usr/lib
mv -v /usr/lib/*gdb.py /usr/share/gdb/auto-load/usr/lib

cd ..
rm -rf gcc-15.2.0

## Ncurses

In [ ]:
tar -xf ncurses-6.5-20250809.tgz
cd ncurses-6.5-20250809

./configure --prefix=/usr           \
            --mandir=/usr/share/man \
            --with-shared           \
            --without-debug         \
            --without-normal        \
            --with-cxx-shared       \
            --enable-pc-files       \
            --with-pkg-config-libdir=/usr/lib/pkgconfig
make

make DESTDIR=$PWD/dest install
install -vm755 dest/usr/lib/libncursesw.so.6.5 /usr/lib
rm -v  dest/usr/lib/libncursesw.so.6.5
sed -e 's/^#if.*XOPEN.*$/#if 1/' \
    -i dest/usr/include/curses.h
cp -av dest/* /

for lib in ncurses form panel menu ; do
    ln -sfv lib${lib}w.so /usr/lib/lib${lib}.so
    ln -sfv ${lib}w.pc    /usr/lib/pkgconfig/${lib}.pc
done

ln -sfv libncursesw.so /usr/lib/libcurses.so

cp -v -R doc -T /usr/share/doc/ncurses-6.5-20250809

cd ..
rm -rf ncurses-6.5-20250809

## Sed

In [ ]:
tar -xf sed-4.9.tar.xz
cd sed-4.9

./configure --prefix=/usr

make
make html

chown -R mgeiger .
su mgeiger -c "PATH=$PATH make check"

make install
install -d -m755           /usr/share/doc/sed-4.9
install -m644 doc/sed.html /usr/share/doc/sed-4.9

cd ..
rm -rf sed-4.9

## Psmisc

In [ ]:
tar -xf psmisc-23.7.tar.xz
cd psmisc-23.7

./configure --prefix=/usr

make

make check

make install

cd ..
rm -rf psmisc-23.7

## Gettext

In [ ]:
tar -xf gettext-0.26.tar.xz
cd gettext-0.26

./configure --prefix=/usr    \
            --disable-static \
            --docdir=/usr/share/doc/gettext-0.26

make

make check

make install
chmod -v 0755 /usr/lib/preloadable_libintl.so

cd ..
rm -rf gettext-0.26

## Bison

In [ ]:
tar -xf bison-3.8.2.tar.xz 
cd bison-3.8.2

./configure --prefix=/usr --docdir=/usr/share/doc/bison-3.8.2

make

make check

make install

cd ..
rm -rf bison-3.8.2

## Grep

In [ ]:
tar -xf grep-3.12.tar.xz
cd grep-3.12

sed -i "s/echo/#echo/" src/egrep.sh

./configure --prefix=/usr

make

make check

make install

cd ..
rm -rf grep-3.12

## Bash

In [ ]:
tar -xf bash-5.3.tar.gz
cd bash-5.3

./configure --prefix=/usr             \
            --without-bash-malloc     \
            --with-installed-readline \
            --docdir=/usr/share/doc/bash-5.3
make

chown -R mgeiger .

LC_ALL=C.UTF-8 su -s /usr/bin/expect tester << "EOF"
set timeout -1
spawn make tests
expect eof
lassign [wait] _ _ _ value
exit $value
EOF

make install

exec /usr/bin/bash --login

cd ../..
rm -rf bash-5.3

## Libtool

In [ ]:
tar -xf libtool-2.5.4.tar.xz 
cd libtool-2.5.4

./configure --prefix=/usr

make

make check

make install

rm -fv /usr/lib/libltdl.a

cd ..
rm -rf libtool-2.5.4

## GDBM

In [ ]:
tar -xf gdbm-latest.tar.gz
cd gdbm-1.26

./configure --prefix=/usr    \
            --disable-static \
            --enable-libgdbm-compat

make

make check

make install

cd ..
rm -rf gdbm-1.26

## Gperf

In [ ]:
tar -xf gperf-3.3.tar.gz
cd gperf-3.3

./configure --prefix=/usr --docdir=/usr/share/doc/gperf-3.3

make

make check
make install

cd ..
rm -rf gperf-3.3

## Expat

In [ ]:
tar -xf expat-2.7.3.tar.xz
cd expat-2.7.3

./configure --prefix=/usr    \
            --disable-static \
            --docdir=/usr/share/doc/expat-2.7.3

make

make check

make install

install -v -m644 doc/*.{html,css} /usr/share/doc/expat-2.7.3

cd ..
rm -rf expat-2.7.3

## Inetutils

In [ ]:
tar -xf inetutils-2.6.tar.xz
cd inetutils-2.6

sed -i 's/def HAVE_TERMCAP_TGETENT/ 1/' telnet/telnet.c

./configure --prefix=/usr        \
            --bindir=/usr/bin    \
            --localstatedir=/var \
            --disable-logger     \
            --disable-whois      \
            --disable-rcp        \
            --disable-rexec      \
            --disable-rlogin     \
            --disable-rsh        \
            --disable-servers

make

make check

make install

mv -v /usr/{,s}bin/ifconfig

cd ..
rm -rf inetutils-2.6

## Less

In [ ]:
tar -xf less-685.tar.gz
cd less-685

./configure --prefix=/usr --sysconfdir=/etc

make

make check

make install

cd ..
rm -rf less-685

## Perl

In [ ]:
tar -xf perl-5.42.0.tar.gz
cd perl-5.42.0

export BUILD_ZLIB=False
export BUILD_BZIP2=0

sh Configure -des                                          \
             -D prefix=/usr                                \
             -D vendorprefix=/usr                          \
             -D privlib=/usr/lib/perl5/5.42/core_perl      \
             -D archlib=/usr/lib/perl5/5.42/core_perl      \
             -D sitelib=/usr/lib/perl5/5.42/site_perl      \
             -D sitearch=/usr/lib/perl5/5.42/site_perl     \
             -D vendorlib=/usr/lib/perl5/5.42/vendor_perl  \
             -D vendorarch=/usr/lib/perl5/5.42/vendor_perl \
             -D man1dir=/usr/share/man/man1                \
             -D man3dir=/usr/share/man/man3                \
             -D pager="/usr/bin/less -isR"                 \
             -D useshrplib                                 \
             -D usethreads

make

TEST_JOBS=$(nproc) make test_harness

make install
unset BUILD_ZLIB BUILD_BZIP2

cd ..
rm -rf perl-5.42.0

## XML::Parser

In [ ]:
tar -xf XML-Parser-2.47.tar.gz
cd XML-Parser-2.47

perl Makefile.PL

make

make test

make install

cd ..
rm -rf XML-Parser-2.47

## Intltool

In [ ]:
tar -xf intltool-0.51.0.tar.gz
cd intltool-0.51.0

sed -i 's:\\\${:\\\$\\{:' intltool-update.in

./configure --prefix=/usr

make

make check

make install
install -v -Dm644 doc/I18N-HOWTO /usr/share/doc/intltool-0.51.0/I18N-HOWTO

cd ..
rm -rf intltool-0.51.0

## Autoconf

In [ ]:
tar -xf autoconf-2.72.tar.xz
cd autoconf-2.72

./configure --prefix=/usr

make

make check

make install

cd ..
rm -rf autoconf-2.72

## Automake

In [ ]:
tar -xf automake-1.18.tar.xz
cd automake-1.18

./configure --prefix=/usr --docdir=/usr/share/doc/automake-1.18.1

make

make -j$(($(nproc)>4?$(nproc):4)) check

make install

cd ..
rm -rf automake-1.18

## OpenSSL

In [ ]:
tar -xf openssl-3.6.0.tar.gz
cd openssl-3.6.0

./config --prefix=/usr         \
         --openssldir=/etc/ssl \
         --libdir=lib          \
         shared                \
         zlib-dynamic

make

HARNESS_JOBS=$(nproc) make test

sed -i '/INSTALL_LIBS/s/libcrypto.a libssl.a//' Makefile
make MANSUFFIX=ssl install

mv -v /usr/share/doc/openssl /usr/share/doc/openssl-3.6.0

cp -vfr doc/* /usr/share/doc/openssl-3.6.0

cd ..
rm -rf openssl-3.6.0

## Libelf

In [ ]:
tar -xf elfutils-0.194.tar.bz2
cd elfutils-0.194

./configure --prefix=/usr        \
            --disable-debuginfod \
            --enable-libdebuginfod=dummy

make

make check

make -C libelf install
install -vm644 config/libelf.pc /usr/lib/pkgconfig
rm /usr/lib/libelf.a

cd ..
rm -rf elfutils-0.194

## Python

In [ ]:
tar -xf Python-3.14.0.tar.xz
cd Python-3.14.0

./configure --prefix=/usr          \
            --enable-shared        \
            --with-system-expat    \
            --enable-optimizations \
            --without-static-libpython

make

make test TESTOPTS="--timeout 120"

make install

cat > /etc/pip.conf << EOF
[global]
root-user-action = ignore
disable-pip-version-check = true
EOF

install -v -dm755 /usr/share/doc/python-3.14.0/html

tar --strip-components=1  \
    --no-same-owner       \
    --no-same-permissions \
    -C /usr/share/doc/python-3.14.0/html \
    -xvf ../python-3.14.0-docs-html.tar.bz2

cd ..
rm -rf Python-3.14.0

## Flit-Core

In [ ]:
tar -xf flit_core-3.12.0.tar.gz
cd flit_core-3.12.0

pip3 wheel -w dist --no-cache-dir --no-build-isolation --no-deps $PWD

pip3 install --no-index --find-links dist flit_core

cd ..
rm -rf flit_core-3.12.0

## Packaging

In [ ]:
tar -xf packaging-25.0.tar.gz 
cd packaging-25.0

pip3 wheel -w dist --no-cache-dir --no-build-isolation --no-deps $PWD

pip3 install --no-index --find-links dist packaging

cd ..
rm -rf packaging-25.0

## Wheel

In [ ]:
tar -xf wheel-0.46.1.tar.gz
cd wheel-0.46.1

pip3 wheel -w dist --no-cache-dir --no-build-isolation --no-deps $PWD

pip3 install --no-index --find-links dist wheel

cd ..
rm -rf wheel-0.46.1

## Setuptools

In [ ]:
tar -xf setuptools-80.9.0.tar.gz
cd setuptools-80.9.0

pip3 wheel -w dist --no-cache-dir --no-build-isolation --no-deps $PWD

pip3 install --no-index --find-links dist setuptools

cd ..
rm -rf setuptools-80.9.0

## Ninja

In [ ]:
tar -xf ninja-1.13.2.tar.gz
cd ninja-1.13.2

export NINJAJOBS=4

sed -i '/int Guess/a \
  int   j = 0;\
  char* jobs = getenv( "NINJAJOBS" );\
  if ( jobs != NULL ) j = atoi( jobs );\
  if ( j > 0 ) return j;\
' src/ninja.cc

python3 configure.py --bootstrap --verbose

install -vm755 ninja /usr/bin/
install -vDm644 misc/bash-completion /usr/share/bash-completion/completions/ninja
install -vDm644 misc/zsh-completion  /usr/share/zsh/site-functions/_ninja

cd ..
rm -rf ninja-1.13.2

## Meson

In [ ]:
tar -xf meson-1.9.1.tar.gz
cd meson-1.9.1

pip3 wheel -w dist --no-cache-dir --no-build-isolation --no-deps $PWD

pip3 install --no-index --find-links dist meson
install -vDm644 data/shell-completions/bash/meson /usr/share/bash-completion/completions/meson
install -vDm644 data/shtar -xf Python-3.14.0.tar.xz
cd Python-3.14.0

./configure --prefix=/usr          \
            --enable-shared        \
            --with-system-expat    \
            --enable-optimizations \
            --without-static-libpython

make

make test TESTOPTS="--timeout 120"

make install

cat > /etc/pip.conf << EOF
[global]
root-user-action = ignore
disable-pip-version-check = true
EOF

install -v -dm755 /usr/share/doc/python-3.14.0/html

tar --strip-components=1  \
    --no-same-owner       \
    --no-same-permissions \
    -C /usr/share/doc/python-3.14.0/html \
    -xvf ../python-3.14.0-docs-html.tar.bz2

cd ..
rm -rf Python-3.14.0ell-completions/zsh/_meson /usr/share/zsh/site-functions/_meson
rm -rf meson-1.9.1

## Kmod

In [ ]:
tar -xf kmod-34.2.tar.xz
cd kmod-34.2

mkdir -p build
cd       build

meson setup --prefix=/usr ..    \
            --buildtype=release \
            -D manpages=false

ninja

ninja install

cd ../..
rm -rf kmod-34.2

## Coreutils

In [ ]:
tar -xf coreutils-9.9.tar.xz
cd coreutils-9.9

patch -Np1 -i ../coreutils-9.9-i18n-1.patch

autoreconf -fv
automake -af
FORCE_UNSAFE_CONFIGURE=1 ./configure \
            --prefix=/usr            \
            --enable-no-install-program=kill,uptime

make

make NON_ROOT_USERNAME=mgeiger check-root

groupadd -g 102 dummy -U mgeiger

chown -R mgeiger . 

su mgeiger -c "PATH=$PATH make -k RUN_EXPENSIVE_TESTS=yes check" \
   < /dev/null

groupdel dummy

make install

mv -v /usr/bin/chroot /usr/sbin
mv -v /usr/share/man/man1/chroot.1 /usr/share/man/man8/chroot.8
sed -i 's/"1"/"8"/' /usr/share/man/man8/chroot.8

cd ..
rm -rf coreutils-9.9

## Diffutils

In [ ]:
tar -xf diffutils-3.12.tar.xz
cd diffutils-3.12

./configure --prefix=/usr

make

make check

make install

cd ..
rm -rf diffutils-3.12

## Gawk

In [ ]:
tar -xf gawk-5.3.2.tar.xz
cd gawk-5.3.2

sed -i 's/extras//' Makefile.in

./configure --prefix=/usr

make

chown -R mgeiger .
su mgeiger -c "PATH=$PATH make check"

rm -f /usr/bin/gawk-5.3.2
make install

ln -sv gawk.1 /usr/share/man/man1/awk.1

install -vDm644 doc/{awkforai.txt,*.{eps,pdf,jpg}} -t /usr/share/doc/gawk-5.3.2

cd ..
rm -rf gawk-5.3.2

## Findutils

In [ ]:
tar -xf findutils-4.10.0.tar.xz
cd findutils-4.10.0

./configure --prefix=/usr --localstatedir=/var/lib/locate

make

chown -R mgeiger .
su mgeiger -c "PATH=$PATH make check"

make install

cd ..
rm -rf findutils-4.10.0

## Groff

In [ ]:
tar -xf groff-1.23.0.tar.gz
cd groff-1.23.0

./configure --prefix=/usr

make

make check

make install

cd ..
rm -rf groff-1.23.0

## GRUB

In [ ]:
tar -xf grub-2.12.tar.xz 
cd grub-2.12

unset {C,CPP,CXX,LD}FLAGS

echo depends bli part_gpt > grub-core/extra_deps.lst

./configure --prefix=/usr     \
            --sysconfdir=/etc \
            --disable-efiemu  \
            --disable-werror

make

make install
mv -v /etc/bash_completion.d/grub /usr/share/bash-completion/completions

cd ..
rm -rf grub-2.12

## Gzip

In [ ]:
tar -xf gzip-1.14.tar.xz
cd gzip-1.14

./configure --prefix=/usr

make

make check

make install

cd ..
rm -rf gzip-1.14

## IPRoute

In [ ]:
tar -xf iproute2-6.17.0.tar.xz
cd iproute2-6.17.0

sed -i /ARPD/d Makefile
rm -fv man/man8/arpd.8

make NETNS_RUN_DIR=/run/netns

make SBINDIR=/usr/sbin install

install -vDm644 COPYING README* -t /usr/share/doc/iproute2-6.17.0

cd ..
rm -rf iproute2-6.17.0

## Kbd

In [ ]:
tar -xf kbd-2.9.0.tar.xz
cd kbd-2.9.0

patch -Np1 -i ../kbd-2.9.0-backspace-1.patch

sed -i '/RESIZECONS_PROGS=/s/yes/no/' configure
sed -i 's/resizecons.8 //' docs/man/man8/Makefile.in

./configure --prefix=/usr --disable-vlock

make

make install

cp -R -v docs/doc -T /usr/share/doc/kbd-2.9.0

cd ..
rm -rf kbd-2.9.0

## Libpipeline

In [ ]:
tar -xf libpipeline-1.5.8.tar.gz
cd libpipeline-1.5.8

./configure --prefix=/usr

make

make install

cd ..
rm -rf libpipeline-1.5.8

## Make

In [ ]:
tar -xf make-4.4.1.tar.gz
cd make-4.4.1

./configure --prefix=/usr

make

chown -R mgeiger .
su mgeiger -c "PATH=$PATH make check"

make install

cd ..
rm -rf make-4.4.1

## Patch

In [ ]:
tar -xf patch-2.8.tar.xz
cd patch-2.8

./configure --prefix=/usr

make

make check

make install

cd ..
rm -rf patch-2.8

## Tar

In [ ]:
tar -xf tar-1.35.tar.xz
cd tar-1.35

FORCE_UNSAFE_CONFIGURE=1  \
./configure --prefix=/usr

make

make check

make install
make -C doc install-html docdir=/usr/share/doc/tar-1.35

cd ..
rm -rf tar-1.35

## Texinfo

In [ ]:
tar -xf texinfo-7.2.tar.xz
cd texinfo-7.2

sed 's/! $output_file eq/$output_file ne/' -i tp/Texinfo/Convert/*.pm

./configure --prefix=/usr

make

make check

make install

make TEXMF=/usr/share/texmf install-tex

pushd /usr/share/info
  rm -v dir
  for f in *
    do install-info $f dir 2>/dev/null
  done
popd

cd ..
rm -rf texinfo-7.2

## Vim

In [ ]:
tar -xf vim-9.1.1934.tar.gz 
cd vim-9.1.1934

echo '#define SYS_VIMRC_FILE "/etc/vimrc"' >> src/feature.h

./configure --prefix=/usr

make

chown -R mgeiger .
sed '/test_plugin_glvs/d' -i src/testdir/Make_all.mak

su mgeiger -c "TERM=xterm-256color LANG=en_US.UTF-8 make -j1 test" \
   &> vim-test.log

make install

ln -sv vim /usr/bin/vi
for L in  /usr/share/man/{,*/}man1/vim.1; do
    ln -sv vim.1 $(dirname $L)/vi.1
done

ln -sv ../vim/vim91/doc /usr/share/doc/vim-9.1.1629

cat > /etc/vimrc << "EOF"
" Begin /etc/vimrc

" Ensure defaults are set before customizing settings, not after
source $VIMRUNTIME/defaults.vim
let skip_defaults_vim=1

set nocompatible
set backspace=2
set mouse=
syntax on
if (&term == "xterm") || (&term == "putty")
  set background=dark
endif

" End /etc/vimrc
EOF

cd ..
rm -rf vim-9.1.1934

## MarkupSafe

In [ ]:
tar -xf markupsafe-3.0.3.tar.gz
cd markupsafe-3.0.3

pip3 wheel -w dist --no-cache-dir --no-build-isolation --no-deps $PWD

pip3 install --no-index --find-links dist Markupsafe

cd ..
rm -rf markupsafe-3.0.3

## Jinja

In [ ]:
tar -xf jinja2-3.1.6.tar.gz
cd jinja2-3.1.6

pip3 wheel -w dist --no-cache-dir --no-build-isolation --no-deps $PWD

pip3 install --no-index --find-links dist Jinja2

cd ..
rm -rf jinja2-3.1.6

## Udev

In [ ]:
tar -xf systemd-258.1.tar.gz
cd systemd-258.1

sed -e 's/GROUP="render"/GROUP="video"/' \
    -e 's/GROUP="sgx", //'               \
    -i rules.d/50-udev-default.rules.in

sed -i '/systemd-sysctl/s/^/#/' rules.d/99-systemd.rules.in

sed -e '/NETWORK_DIRS/s/systemd/udev/' \
    -i src/libsystemd/sd-network/network-util.h

mkdir -p build
cd       build

meson setup ..                  \
      --prefix=/usr             \
      --buildtype=release       \
      -D mode=release           \
      -D dev-kvm-mode=0660      \
      -D link-udev-shared=false \
      -D logind=false           \
      -D vconsole=false

export udev_helpers=$(grep "'name' :" ../src/udev/meson.build | \
                      awk '{print $3}' | tr -d ",'" | grep -v 'udevadm')

ninja udevadm systemd-hwdb                                           \
      $(ninja -n | grep -Eo '(src/(lib)?udev|rules.d|hwdb.d)/[^ ]*') \
      $(realpath libudev.so --relative-to .)                         \
      $udev_helpers

install -vm755 -d {/usr/lib,/etc}/udev/{hwdb.d,rules.d,network}
install -vm755 -d /usr/{lib,share}/pkgconfig
install -vm755 udevadm                             /usr/bin/
install -vm755 systemd-hwdb                        /usr/bin/udev-hwdb
ln      -svfn  ../bin/udevadm                      /usr/sbin/udevd
cp      -av    libudev.so{,*[0-9]}                 /usr/lib/
install -vm644 ../src/libudev/libudev.h            /usr/include/
install -vm644 src/libudev/*.pc                    /usr/lib/pkgconfig/
install -vm644 src/udev/*.pc                       /usr/share/pkgconfig/
install -vm644 ../src/udev/udev.conf               /etc/udev/
install -vm644 rules.d/* ../rules.d/README         /usr/lib/udev/rules.d/
install -vm644 $(find ../rules.d/*.rules \
                      -not -name '*power-switch*') /usr/lib/udev/rules.d/
install -vm644 hwdb.d/*  ../hwdb.d/{*.hwdb,README} /usr/lib/udev/hwdb.d/
install -vm755 $udev_helpers                       /usr/lib/udev
install -vm644 ../network/99-default.link          /usr/lib/udev/network

tar -xvf ../../udev-lfs-20230818.tar.xz
make -f udev-lfs-20230818/Makefile.lfs install

tar -xf ../../systemd-man-pages-257.8.tar.xz                            \
    --no-same-owner --strip-components=1                              \
    -C /usr/share/man --wildcards '*/udev*' '*/libudev*'              \
                                  '*/systemd.link.5'                  \
                                  '*/systemd-'{hwdb,udevd.service}.8

sed 's|systemd/network|udev/network|'                                 \
    /usr/share/man/man5/systemd.link.5                                \
  > /usr/share/man/man5/udev.link.5

sed 's/systemd\(\\\?-\)/udev\1/' /usr/share/man/man8/systemd-hwdb.8   \
                               > /usr/share/man/man8/udev-hwdb.8

sed 's|lib.*udevd|sbin/udevd|'                                        \
    /usr/share/man/man8/systemd-udevd.service.8                       \
  > /usr/share/man/man8/udevd.8

rm /usr/share/man/man*/systemd*

unset udev_helpers

udev-hwdb update

cd ..
rm -rf systemd-258.1

## Git

In [ ]:
tar -xf git-2.5.0.tar.xz
cd git-2.5.0

./configure --prefix=/usr                   \
            --with-gitconfig=/etc/gitconfig \
            --with-python=python3           \
            --with-libpcre2                 &&
make

make html

make man

make perllibdir=/usr/lib/perl5/5.42/site_perl install

make install-man

make htmldir=/usr/share/doc/git-2.52.0 install-html

tar -xf ../git-manpages-2.52.0.tar.xz \
    -C /usr/share/man --no-same-owner --no-overwrite-dir

mkdir -vp   /usr/share/doc/git-2.52.0 &&
tar   -xf   ../git-htmldocs-2.52.0.tar.xz \
      -C    /usr/share/doc/git-2.52.0 --no-same-owner --no-overwrite-dir &&

find        /usr/share/doc/git-2.52.0 -type d -exec chmod 755 {} \; &&
find        /usr/share/doc/git-2.52.0 -type f -exec chmod 644 {} \;

mkdir -vp /usr/share/doc/git-2.52.0/man-pages/{html,text}         &&
mv        /usr/share/doc/git-2.52.0/{git*.adoc,man-pages/text}     &&
mv        /usr/share/doc/git-2.52.0/{git*.,index.,man-pages/}html &&

mkdir -vp /usr/share/doc/git-2.52.0/technical/{html,text}         &&
mv        /usr/share/doc/git-2.52.0/technical/{*.adoc,text}        &&
mv        /usr/share/doc/git-2.52.0/technical/{*.,}html           &&

mkdir -vp /usr/share/doc/git-2.52.0/howto/{html,text}             &&
mv        /usr/share/doc/git-2.52.0/howto/{*.adoc,text}            &&
mv        /usr/share/doc/git-2.52.0/howto/{*.,}html               &&

sed -i '/^<a href=/s|howto/|&html/|' /usr/share/doc/git-2.52.0/howto-index.html &&
sed -i '/^\* link:/s|howto/|&html/|' /usr/share/doc/git-2.52.0/howto-index.adoc

cd ..
rm -rf git-2.5.0

## Man-DB

In [ ]:
tar -xf man-db-2.13.1.tar.xz
cd man-db-2.13.1

./configure --prefix=/usr                         \
            --docdir=/usr/share/doc/man-db-2.13.1 \
            --sysconfdir=/etc                     \
            --disable-setuid                      \
            --enable-cache-owner=bin              \
            --with-browser=/usr/bin/lynx          \
            --with-vgrind=/usr/bin/vgrind         \
            --with-grap=/usr/bin/grap             \
            --with-systemdtmpfilesdir=            \
            --with-systemdsystemunitdir=

make

make check

make install

cd ..
rm -rf man-db-2.13.1

## Procps-ng

In [ ]:
tar -xf procps-ng-4.0.5.tar.xz 
cd procps-ng-4.0.5

./configure --prefix=/usr                           \
            --docdir=/usr/share/doc/procps-ng-4.0.5 \
            --disable-static                        \
            --disable-kill                          \
            --enable-watch8bit

make

chown -R mgeiger .
su mgeiger -c "PATH=$PATH make check"

make install

cd ..
rm -rf procps-ng-4.0.5

## Util-linux

In [ ]:
tar -xf util-linux-2.41.2.tar.xz
cd util-linux-2.41.2

./configure --bindir=/usr/bin     \
            --libdir=/usr/lib     \
            --runstatedir=/run    \
            --sbindir=/usr/sbin   \
            --disable-chfn-chsh   \
            --disable-login       \
            --disable-nologin     \
            --disable-su          \
            --disable-setpriv     \
            --disable-runuser     \
            --disable-pylibmount  \
            --disable-liblastlog2 \
            --disable-static      \
            --without-python      \
            --without-systemd     \
            --without-systemdsystemunitdir        \
            ADJTIME_PATH=/var/lib/hwclock/adjtime \
            --docdir=/usr/share/doc/util-linux-2.41.1

make

bash tests/run.sh --srcdir=$PWD --builddir=$PWD

touch /etc/fstab
chown -R mgeiger .
su mgeiger -c "make -k check"

make install

cd ..
rm -rf util-linux-2.41.2

## E2fsprogs

In [ ]:
tar -xf e2fsprogs-1.47.3.tar.gz 
cd e2fsprogs-1.47.3

mkdir -v build
cd build

../configure --prefix=/usr       \
             --sysconfdir=/etc   \
             --enable-elf-shlibs \
             --disable-libblkid  \
             --disable-libuuid   \
             --disable-uuidd     \
             --disable-fsck

make

make check

make install

rm -fv /usr/lib/{libcom_err,libe2p,libext2fs,libss}.a

gunzip -v /usr/share/info/libext2fs.info.gz
install-info --dir-file=/usr/share/info/dir /usr/share/info/libext2fs.info

makeinfo -o      doc/com_err.info ../lib/et/com_err.texinfo
install -v -m644 doc/com_err.info /usr/share/info
install-info --dir-file=/usr/share/info/dir /usr/share/info/com_err.info

sed 's/metadata_csum_seed,//' -i /etc/mke2fs.conf

cd ../..
rm -rf e2fsprogs-1.47.3

## Sysklogd

In [ ]:
tar -xf sysklogd-2.7.2.tar.gz
cd sysklogd-2.7.2

./configure --prefix=/usr      \
            --sysconfdir=/etc  \
            --runstatedir=/run \
            --without-logger   \
            --disable-static   \
            --docdir=/usr/share/doc/sysklogd-2.7.2

make

make install

cat > /etc/syslog.conf << "EOF"
# Begin /etc/syslog.conf

auth,authpriv.* -/var/log/auth.log
*.*;auth,authpriv.none -/var/log/sys.log
daemon.* -/var/log/daemon.log
kern.* -/var/log/kern.log
mail.* -/var/log/mail.log
user.* -/var/log/user.log
*.emerg *

# Do not open any internet ports.
secure_mode 2

# End /etc/syslog.conf
EOF

cd ..
rm -rf sysklogd-2.7.2

## SysVinit

In [ ]:
tar -xf sysvinit-3.14.tar.xz 
cd sysvinit-3.14

patch -Np1 -i ../sysvinit-3.14-consolidated-1.patch

make 

make install

cd ..
rm -rf sysvinit-3.14

## Libpsl

In [ ]:
tar -xf libpsl-0.21.5.tar.gz
cd libpsl-0.21.5

mkdir build &&
cd    build &&

meson setup --prefix=/usr --buildtype=release &&

ninja

ninja install

cd ../..
rm -rf libpsl-0.21.5

## Curl

In [ ]:
tar -xf curl-8.17.0.tar.xz
cd curl-8.17.0

sed -i 's/2F 5C/%2F %5C/' scripts/wcurl

./configure --prefix=/usr    \
            --disable-static \
            --with-openssl   \
            --with-ca-path=/etc/ssl/certs &&
make

make install &&

rm -rf docs/examples/.deps &&

find docs \( -name Makefile\* -o  \
             -name \*.1       -o  \
             -name \*.3       -o  \
             -name CMakeLists.txt \) -delete &&

cp -v -R docs -T /usr/share/doc/curl-8.17.0

cd ..
rm -rf curl-8.17.0

In [ ]:
tar -xzf openssh-9.8p1.tar.gz

cd openssh-9.8p1

./configure --prefix=/usr \
            --sysconfdir=/etc/ssh \
            --with-privsep-path=/var/lib/sshd \
            --with-default-path=/usr/bin \
            --with-superuser-path=/usr/sbin:/usr/bin \
            --with-pid-dir=/run

make

make install

# Cleaning Up

FINALLY!!!!!!

In [ ]:
rm -rf /tmp/{*,.*}

find /usr/lib /usr/libexec -name \*.la -delete

find /usr -depth -name $(uname -m)-lfs-linux-gnu\* | xargs rm -rf

userdel -r mgeiger

### Bootscripts

In [ ]:
tar -xf lfs-bootscripts-20250827.tar.xz
cd lfs-bootscripts-20250827

make install

cd ..
rm -rf lfs-bootscripts-20250827

## DHCP

In [ ]:
tar xf dhcpcd-10.0.6.tar.xz
cd dhcpcd-10.0.6

./configure --prefix=/usr \
            --sysconfdir=/etc \
            --libexecdir=/usr/lib/dhcpcd \
            --dbdir=/var/lib/dhcpcd \
            --runstatedir=/run \
            --disable-privsep

make
make install

cd ..
rm -rf dhcpcd-10.0.6

## Adding Networking

In [ ]:
cat > /etc/sysconfig/ifconfig.eth0 << "EOF"
ONBOOT=yes
IFACE=eth0
SERVICE=dhcpcd
DHCP_START="-b -q"
DHCP_STOP="-k"
EOF

# DNS (optional fallback)
cat > /etc/resolv.conf << "EOF"
nameserver 8.8.8.8
nameserver 8.8.4.4
EOF

# Booting

In [ ]:
cat > /etc/fstab << "EOF"
# Begin /etc/fstab

# file system  mount-point    type     options             dump  fsck
#                                                                order

/dev/sda3     /              ext4    defaults            1     1
/dev/sda2     swap           swap     pri=1               0     0
proc           /proc          proc     nosuid,noexec,nodev 0     0
sysfs          /sys           sysfs    nosuid,noexec,nodev 0     0
devpts         /dev/pts       devpts   gid=5,mode=620      0     0
tmpfs          /run           tmpfs    defaults            0     0
devtmpfs       /dev           devtmpfs mode=0755,nosuid    0     0
tmpfs          /dev/shm       tmpfs    nosuid,nodev        0     0
cgroup2        /sys/fs/cgroup cgroup2  nosuid,noexec,nodev 0     0

# End /etc/fstab
EOF

## Getting Linux

### Foreword

In the next section there is a command ```make menuconfig```. You need to select the following options



In [ ]:
General setup --->
  [ ] Compile the kernel with warnings as errors                        [WERROR]
  [*] Initial RAM filesystem and RAM disk (initramfs/initrd) support
                                                              [BLK_DEV_INITRD]
  CPU/Task time and stats accounting --->
    [*] Pressure stall information tracking                                [PSI]
    [ ]   Require boot parameter to enable pressure stall information tracking
                                                       [PSI_DEFAULT_DISABLED]
  < > Enable kernel headers through /sys/kernel/kheaders.tar.xz      [IKHEADERS]
  [*] Control Group support --->                                       [CGROUPS]
    [*] Memory controller                                                [MEMCG]
  [ ] Configure standard kernel features (expert users) --->            [EXPERT]

Processor type and features --->
  [*] Build a relocatable kernel                                   [RELOCATABLE]
    [*] Randomize the address of the kernel image (KASLR)       [RANDOMIZE_BASE]

General architecture-dependent options --->
  [*] Stack Protector buffer overflow detection                 [STACKPROTECTOR]
    [*] Strong Stack Protector                           [STACKPROTECTOR_STRONG]

Device Drivers --->
  Generic Driver Options --->
    [ ] Support for uevent helper                                [UEVENT_HELPER]
    [*] Maintain a devtmpfs filesystem to mount at /dev               [DEVTMPFS]
      [*] Automount devtmpfs at /dev, after the kernel mounted rootfs
                                                              [DEVTMPFS_MOUNT]
  [*] Block devices --->                                              [BLK_DEV]
    <*> RAM block device support                                  [BLK_DEV_RAM]
      (16)  Default number of RAM disks                     [BLK_DEV_RAM_COUNT]
      (65536) Default RAM disk size (kbytes)                 [BLK_DEV_RAM_SIZE]
  Network device support --->
    Ethernet driver support --->
      Intel devices --->
        <*> Intel(R) PRO/1000 Gigabit Ethernet support                  [E1000]
    USB Network Adapters --->
      <*> Multi-purpose USB Networking Framework                    [USB_USBNET]
  Firmware Drivers --->
    [*] Mark VGA/VBE/EFI FB as generic system framebuffer       [SYSFB_SIMPLEFB]
  Graphics support --->
    <*> Direct Rendering Manager (XFree86 4.1.0 and higher DRI support)   [DRM]
      [*] Display a user-friendly message when a kernel panic occurs
                                                                    [DRM_PANIC]
        (kmsg) Panic screen formatter                       [DRM_PANIC_SCREEN]
      Supported DRM clients --->
        [*] Enable legacy fbdev support for your modesetting driver
                                                          [DRM_FBDEV_EMULATION]
    Drivers for system framebuffers --->
      <*> Simple framebuffer driver                              [DRM_SIMPLEDRM]
    Console display driver support --->
      [*] Framebuffer Console support                    [FRAMEBUFFER_CONSOLE]

File systems --->
  <*> The Extended 4 (ext4) filesystem                               [EXT4_FS]
  Miscellaneous filesystems --->
    <*> SquashFS 4.0 - Squashed file system support                  [SQUASHFS]
      [*] Squashfs XATTR support                              [SQUASHFS_XATTR]
      [*] Include support for ZLIB compressed file systems     [SQUASHFS_ZLIB]
      [*] Include support for XZ compressed file systems         [SQUASHFS_XZ]
  Pseudo filesystems --->
    [*] /proc file system support                                    [PROC_FS]
    [*] sysfs file system support                                      [SYSFS]

Networking support --->
  Networking options --->
    [*] TCP/IP networking                                               [INET]

## Compiling the Kernel

In [ ]:
tar -xf linux-6.17.9.tar.xz
cd linux-6.17.9

make mrproper

make menuconfig # <--- This part

make

make modules_install

mount /boot

cp -iv arch/x86/boot/bzImage /boot/vmlinuz-6.17.9-lfs-12.4

cp -iv System.map /boot/System.map-6.17.9

cp -iv .config /boot/config-6.17.9

cp -r Documentation -T /usr/share/doc/linux-6.17.9

cd ..
rm -rf linux-6.17.9

install -v -m755 -d /etc/modprobe.d
cat > /etc/modprobe.d/usb.conf << "EOF"
# Begin /etc/modprobe.d/usb.conf

install ohci_hcd /sbin/modprobe ehci_hcd ; /sbin/modprobe -i ohci_hcd ; true
install uhci_hcd /sbin/modprobe ehci_hcd ; /sbin/modprobe -i uhci_hcd ; true

# End /etc/modprobe.d/usb.conf
EOF

## Squashfs

In [ ]:
tar -xf squashfs-tools-4.7.4.tar.gz
cd squashfs-tools-4.7.4/squashfs-tools

make GZIP_SUPPORT=1 XZ_SUPPORT=1 LZO_SUPPORT=0 LZMA_XZ_SUPPORT=0 LZ4_SUPPORT=0 ZSTD_SUPPORT=0 XATTR_SUPPORT=1

make install PREFIX=/usr

cd ../..
rm -rf squashfs-tools-4.7.4

## Libburn

In [ ]:
tar -xf libburn_1.5.4.orig.tar.gz
cd libburn-1.5.4

sed -i 's/catch_int ()/catch_int (int signum)/' test/poll.c

./configure --prefix=/usr --disable-static &&
make

cd ..
rm -rf libburn-1.5.4

## Libisoburn (provides xorriso)

In [ ]:
tar -xf libburn_1.5.4.orig.tar.gz
cd libburn-1.5.4

./configure --prefix=/usr --disable-static
make
make install

cd ..
rm -rf libburn-1.5.4

## Libisofs

In [ ]:
tar -xf libisofs-1.5.6.pl01.tar.gz
cd libisofs-1.5.6

./configure --prefix=/usr --disable-static
make
make install

cd ..
rm -rf libisofs-1.5.6

## Libisoburn

In [ ]:
tar -xf libisoburn-1.5.6.tar.gz
cd libisoburn-1.5.6

./configure --prefix=/usr --disable-static
make
make install

cd ..
rm -rf libisoburn-1.5.6

## Cpio

In [ ]:
tar -xf cpio-2.15.tar.bz2
cd cpio-2.15

./configure --prefix=/usr \
            --enable-mt   \
            --with-rmt=/usr/libexec/rmt

make
make install

cd ..
rm -rf cpio-2.15

## Setup Networking

This will give us the basic Ethernet (enp0s3) configuration so we can access the internet automatically.

In [ ]:
#!/usr/bin/bash

#==============================================================================
# Network Auto-Start Configuration
#==============================================================================
echo "Configuring automatic network startup..."

# Detect network interface (usually enp0s3 in VirtualBox, eth0 elsewhere)
IFACE=$(ip link show | grep -v "lo:" | grep "state" | head -1 | awk '{print $2}' | tr -d ':')

if [ -z "$IFACE" ]; then
    echo "Warning: Could not detect network interface, using default 'enp0s3'"
    IFACE="enp0s3"
fi

echo "Network interface detected: $IFACE"

# Create DHCP network configuration
cat > /etc/sysconfig/ifconfig.$IFACE << NETEOF
ONBOOT=yes
IFACE=$IFACE
SERVICE=dhcpcd
DHCP_START="-b -q"
DHCP_STOP="-k"
NETEOF

# Clean up any corrupted config files
rm -f /etc/sysconfig/ifconfig-rc.site 2>/dev/null
touch /etc/sysconfig/ifconfig-rc.site

# Set DNS resolvers
cat > /etc/resolv.conf << "DNSEOF"
nameserver 8.8.8.8
nameserver 8.8.4.4
DNSEOF

echo "✓ Network configuration created for $IFACE"
echo "  - DHCP enabled"
echo "  - Auto-start on boot enabled"
echo "  - DNS configured (8.8.8.8, 8.8.4.4)"


## Network Auto-Start Configuration

Configure the network to start automatically using DHCP. This ensures networking is available when SSH starts.

In [ ]:
# 1. Create Config
cat > /etc/sysconfig/ifconfig-rc.site << "EOF"
# Optional parameters for ifup/ifdown
EOF

# 2. Ensure the DHCP config is correct
cat > /etc/sysconfig/ifconfig.enp0s3 << "EOF"
ONBOOT=yes
IFACE=enp0s3
SERVICE=dhcpcd
DHCP_START="-b -q"
DHCP_STOP="-k"
EOF

# 3. Create the network boot script symlink (if not already present)
cd /etc/rc.d/rc3.d
ln -sf ../init.d/network S10network 2>/dev/null

# 4. Test the network script works
/etc/rc.d/init.d/network restart

In [ ]:
#==============================================================================
# SSH Init Script and Auto-Start Configuration
#==============================================================================
echo "Creating SSH init script..."

cat > /etc/rc.d/init.d/sshd << "SSHDEOF"
#!/bin/bash
# Begin sshd

### BEGIN INIT INFO
# Provides:            sshd
# Required-Start:      $network
# Should-Start:
# Required-Stop:       $network
# Should-Stop:
# Default-Start:       3 4 5
# Default-Stop:        0 1 2 6
# Short-Description:   OpenSSH Server Daemon
# Description:         OpenSSH Server Daemon - Secure Shell
### END INIT INFO

. /lib/lsb/init-functions

SSHD=/usr/sbin/sshd
PIDFILE=/run/sshd.pid

case "${1}" in
   start)
      log_info_msg "Starting SSH Server..."
      
      # Generate host keys if missing
      if [ ! -f /etc/ssh/ssh_host_rsa_key ]; then
         log_info_msg "Generating SSH host keys..."
         ssh-keygen -A
      fi
      
      start_daemon $SSHD
      evaluate_retval
      ;;

   stop)
      log_info_msg "Stopping SSH Server..."
      killproc $SSHD
      evaluate_retval
      ;;

   reload)
      log_info_msg "Reloading SSH Server configuration..."
      killproc -HUP $SSHD
      evaluate_retval
      ;;

   restart)
      ${0} stop
      sleep 1
      ${0} start
      ;;

   status)
      statusproc $SSHD
      ;;

   *)
      echo "Usage: ${0} {start|stop|reload|restart|status}"
      exit 1
      ;;
esac

exit 0
# End sshd
SSHDEOF

# Make executable
chmod 755 /etc/rc.d/init.d/sshd

# Link to start on boot (after network at S30)
ln -sf /etc/rc.d/init.d/sshd /etc/rc.d/rc3.d/S30sshd
ln -sf /etc/rc.d/init.d/sshd /etc/rc.d/rc4.d/S30sshd
ln -sf /etc/rc.d/init.d/sshd /etc/rc.d/rc5.d/S30sshd

# Link to stop on shutdown
ln -sf /etc/rc.d/init.d/sshd /etc/rc.d/rc0.d/K30sshd
ln -sf /etc/rc.d/init.d/sshd /etc/rc.d/rc1.d/K30sshd
ln -sf /etc/rc.d/init.d/sshd /etc/rc.d/rc2.d/K30sshd
ln -sf /etc/rc.d/init.d/sshd /etc/rc.d/rc6.d/K30sshd

echo "SSH init script created and enabled for auto-start"


## SSH Init Script

Create the SysVinit script for SSH daemon auto-start.

# Using GRUB to Set Up the Boot Process

In [ ]:
grub-install /dev/sda

cat > /boot/grub/grub.cfg << "EOF"
# Begin /boot/grub/grub.cfg
set default=0
set timeout=5

insmod part_gpt
insmod ext2
set root=(hd0,2)
set gfxpayload=1024x768x32

menuentry "GNU/Linux, Linux 6.16.1-lfs-12.4" {
        linux   /boot/vmlinuz-6.16.1-lfs-12.4 root=/dev/sda2 ro
}
EOF

Check the progress

In [ ]:
lsblk

mount | grep " / "

# Restarting

In [ ]:
sudo umount -v $LFS/dev/pts
sudo mountpoint -q $LFS/dev/shm && sudo umount -v $LFS/dev/shm
sudo umount -v $LFS/dev
sudo umount -v $LFS/run
sudo umount -v $LFS/proc
sudo umount -v $LFS/sys

sudo umount -v $LFS/home
sudo umount -v $LFS

# Making a Distributable ISO

> ⚠️ **PREREQUISITE**: Complete this section AFTER you have successfully booted SchmitziOS at least once. This ensures your base system works before packaging it.

This section will create a bootable ISO that can be used to install SchmitziOS on any computer.

## Overview

The ISO contains:
- **GRUB** - Boots the ISO on BIOS systems
- **Initramfs** - Minimal environment that mounts the squashfs
- **filesystem.squashfs** - Your compressed SchmitziOS system
- **Installer script** - Simple text-based installer

```
┌─────────────────────────────────────────────┐
│  SchmitziOS-1.0.iso                         │
│  ├── boot/                                  │
│  │   ├── grub/grub.cfg                      │
│  │   ├── vmlinuz                            │
│  │   └── initramfs.img                      │
│  ├── live/                                  │
│  │   └── filesystem.squashfs               │
│  └── install.sh                             │
└─────────────────────────────────────────────┘
```

The required packages (squashfs-tools, libisoburn, cpio) were already built earlier.

## Step 1: Create Working Directories

Log in to your booted SchmitziOS as root and run:

In [ ]:
# Create ISO staging area
mkdir -p /root/iso/{boot/grub,live}

# Create initramfs build area
mkdir -p /root/initramfs/{bin,sbin,etc,proc,sys,dev,newroot,run,lib,lib64,usr/lib,mnt/cdrom,mnt/squash}

## Step 2: Build Busybox for Initramfs

We use a statically compiled Busybox for a reliable, small initramfs.

In [ ]:
tar -xf busybox-1.36.1.tar.bz2
cd busybox-1.36.1

make defconfig

# Enable static build
sed -i 's/# CONFIG_STATIC is not set/CONFIG_STATIC=y/' .config

make
make install

# Copy busybox to initramfs
cp -a _install/* /root/initramfs/

cd ..
rm -rf busybox-1.36.1

## Step 3: Create the Init Script

This script runs when the ISO boots. It finds and mounts the squashfs.

In [ ]:
cat > /root/initramfs/init << 'INITEOF'
#!/bin/sh
# SchmitziOS Initramfs Init Script

# Mount essential filesystems
mount -t proc none /proc
mount -t sysfs none /sys
mount -t devtmpfs none /dev

echo "========================================"
echo "      SchmitziOS Live Boot"
echo "========================================"

# Wait for devices to settle
sleep 3

# Find the ISO/USB device containing our squashfs
echo "Looking for SchmitziOS filesystem..."

FOUND=""
for device in /dev/sr0 /dev/sda1 /dev/sda /dev/sdb1 /dev/sdb /dev/sdc1 /dev/nvme0n1p1; do
    if [ -b "$device" ]; then
        if mount -o ro "$device" /mnt/cdrom 2>/dev/null; then
            if [ -f /mnt/cdrom/live/filesystem.squashfs ]; then
                FOUND="$device"
                echo "Found SchmitziOS on $device"
                break
            fi
            umount /mnt/cdrom
        fi
    fi
done

if [ -z "$FOUND" ]; then
    echo "ERROR: Could not find SchmitziOS filesystem!"
    echo "Dropping to emergency shell..."
    exec /bin/sh
fi

# Mount the squashfs
echo "Mounting squashfs filesystem..."
mount -t squashfs -o loop /mnt/cdrom/live/filesystem.squashfs /mnt/squash

if [ $? -ne 0 ]; then
    echo "ERROR: Failed to mount squashfs!"
    exec /bin/sh
fi

# Prepare to switch root
mkdir -p /mnt/squash/mnt/cdrom
mount --move /mnt/cdrom /mnt/squash/mnt/cdrom

# Clean up
umount /proc
umount /sys
umount /dev

echo "Switching to SchmitziOS..."
exec switch_root /mnt/squash /sbin/init
INITEOF

chmod +x /root/initramfs/init

## Step 4: Create the Initramfs Image

In [ ]:
cd /root/initramfs
find . | cpio -o -H newc | gzip > /root/iso/boot/initramfs.img

echo "Initramfs created!"
ls -lh /root/iso/boot/initramfs.img

## Step 5: Create the Squashfs Filesystem

This compresses your entire SchmitziOS system.

In [ ]:
mksquashfs / /root/iso/live/filesystem.squashfs \
    -e /root/iso \
    -e /root/initramfs \
    -e /sources \
    -e /tmp \
    -e /var/tmp \
    -e /var/cache \
    -e /proc \
    -e /sys \
    -e /dev \
    -e /run \
    -e /mnt \
    -comp xz \
    -Xbcj x86

echo "Squashfs created! Size:"
ls -lh /root/iso/live/filesystem.squashfs

## Pre-ISO SSH Verification

**CRITICAL:** Test SSH configuration before creating the ISO to ensure everything is properly set up.

In [ ]:
#!/usr/bin/bash
 
echo "========================================"
echo "  Testing SSH Configuration"
echo "========================================"
echo ""

PASS=0
FAIL=0

# Check sshd binary
if [ -f /usr/sbin/sshd ]; then
    echo "✓ sshd binary found"
    PASS=$((PASS+1))
else
    echo "✗ sshd binary NOT found - SSH will not work!"
    FAIL=$((FAIL+1))
fi

# Check SSH config
if [ -f /etc/ssh/sshd_config ]; then
    echo "✓ sshd_config found"
    PASS=$((PASS+1))
else
    echo "✗ sshd_config NOT found"
    FAIL=$((FAIL+1))
fi

# Check host keys
if [ -f /etc/ssh/ssh_host_rsa_key ]; then
    echo "✓ SSH host keys found"
    PASS=$((PASS+1))
else
    echo "! SSH host keys not found (will be generated on first boot)"
fi

# Check init script
if [ -f /etc/rc.d/init.d/sshd ] && [ -x /etc/rc.d/init.d/sshd ]; then
    echo "✓ SSH init script found and executable"
    PASS=$((PASS+1))
else
    echo "✗ SSH init script NOT found or not executable"
    FAIL=$((FAIL+1))
fi

# Check boot link
if [ -L /etc/rc.d/rc3.d/S30sshd ]; then
    echo "✓ SSH set to start on boot"
    PASS=$((PASS+1))
else
    echo "✗ SSH NOT set to start on boot"
    FAIL=$((FAIL+1))
fi

# Check root password
if grep -q '^root:\$' /etc/shadow; then
    echo "✓ Root password is set"
    PASS=$((PASS+1))
else
    echo "✗ WARNING: Root password may not be set!"
    FAIL=$((FAIL+1))
fi

# Check network config
if ls /etc/sysconfig/ifconfig.* >/dev/null 2>&1; then
    echo "✓ Network configuration found"
    PASS=$((PASS+1))
else
    echo "! Network configuration may be missing"
fi

echo ""
echo "========================================"
echo "  Test Results: $PASS passed, $FAIL failed"
echo "========================================"

if [ $FAIL -gt 0 ]; then
    echo ""
    echo "⚠️  CRITICAL: Some SSH components are missing!"
    echo "    SSH will NOT work after booting the ISO."
    echo "    Fix the issues above before continuing."
    echo ""
    read -p "Continue anyway? (y/N): " -n 1 -r
    echo
    if [[ ! $REPLY =~ ^[Yy]$ ]]; then
        echo "Aborting. Fix SSH setup and try again."
        exit 1
    fi
else
    echo ""
    echo "✓ SSH Configuration Summary:"
    echo "  - SSH daemon will start automatically on boot"
    echo "  - Root login via SSH is enabled"
    echo "  - Password authentication is enabled"
    echo "  - Default credentials: root / password"
    echo ""
    echo "  ⚠️  SECURITY WARNING: Change default password after first boot!"
    echo ""
fi


## Step 6: Copy the Kernel

In [ ]:
cp /boot/vmlinuz* /root/iso/boot/vmlinuz

## Step 7: Create the Installer Script

In [ ]:
cat > /root/iso/install.sh << 'INSTALLEOF'
#!/bin/bash
#
# SchmitziOS Installer

set -e

echo "========================================"
echo "      SchmitziOS Installer"
echo "========================================"
echo ""
echo "WARNING: This will ERASE the target disk!"
echo ""

# Show available disks
echo "Available disks:"
echo "----------------"
lsblk -d -o NAME,SIZE,MODEL | grep -v loop
echo ""

read -p "Enter target disk (e.g., sda): " TARGET_DISK
TARGET="/dev/${TARGET_DISK}"

if [ ! -b "$TARGET" ]; then
    echo "Error: $TARGET is not a valid block device!"
    exit 1
fi

echo ""
echo "Target disk: $TARGET"
lsblk "$TARGET"
echo ""
read -p "ALL DATA ON $TARGET WILL BE DESTROYED! Continue? (yes/no): " CONFIRM

if [ "$CONFIRM" != "yes" ]; then
    echo "Aborted."
    exit 1
fi

echo ""
echo "==> Creating partitions..."

# Create partitions: 1MB BIOS boot, 512MB boot, rest for root
parted -s "$TARGET" mklabel gpt
parted -s "$TARGET" mkpart bios_boot 1MiB 2MiB
parted -s "$TARGET" set 1 bios_grub on
parted -s "$TARGET" mkpart boot ext4 2MiB 514MiB
parted -s "$TARGET" mkpart root ext4 514MiB 100%

# Wait for partitions to appear
sleep 2
partprobe "$TARGET"
sleep 1

BOOT_PART="${TARGET}2"
ROOT_PART="${TARGET}3"

# Handle nvme naming
if [[ "$TARGET" == *"nvme"* ]]; then
    BOOT_PART="${TARGET}p2"
    ROOT_PART="${TARGET}p3"
fi

echo "==> Formatting partitions..."
mkfs.ext4 -F -L SCHMITZI_BOOT "$BOOT_PART"
mkfs.ext4 -F -L SCHMITZI_ROOT "$ROOT_PART"

echo "==> Mounting partitions..."
mkdir -p /mnt/target
mount "$ROOT_PART" /mnt/target
mkdir -p /mnt/target/boot
mount "$BOOT_PART" /mnt/target/boot

echo "==> Installing SchmitziOS (this may take a while)..."

# Find where we booted from
SQUASHFS=""
for dir in /mnt/cdrom /run/live/medium /cdrom; do
    if [ -f "$dir/live/filesystem.squashfs" ]; then
        SQUASHFS="$dir/live/filesystem.squashfs"
        break
    fi
done

if [ -z "$SQUASHFS" ]; then
    echo "Error: Could not find filesystem.squashfs!"
    exit 1
fi

# Extract squashfs to target
unsquashfs -f -d /mnt/target "$SQUASHFS"

echo "==> Generating fstab..."
ROOT_UUID=$(blkid -s UUID -o value "$ROOT_PART")
BOOT_UUID=$(blkid -s UUID -o value "$BOOT_PART")

cat > /mnt/target/etc/fstab << FSTABEOF
# /etc/fstab - SchmitziOS
UUID=$ROOT_UUID  /        ext4    defaults          1      1
UUID=$BOOT_UUID  /boot    ext4    defaults          1      2
proc             /proc    proc    nosuid,noexec,nodev 0      0
sysfs            /sys     sysfs   nosuid,noexec,nodev 0      0
devpts           /dev/pts devpts  gid=5,mode=620      0      0
tmpfs            /run     tmpfs   defaults            0      0
devtmpfs         /dev     devtmpfs mode=0755,nosuid   0      0
FSTABEOF

echo "==> Installing GRUB bootloader..."

# Mount necessary filesystems for chroot
mount --bind /dev /mnt/target/dev
mount --bind /dev/pts /mnt/target/dev/pts
mount -t proc proc /mnt/target/proc
mount -t sysfs sysfs /mnt/target/sys

# Install GRUB
chroot /mnt/target grub-install --target=i386-pc "$TARGET"

# Create GRUB config
chroot /mnt/target bash -c "cat > /boot/grub/grub.cfg << GRUBEOF
set default=0
set timeout=5

insmod part_gpt
insmod ext2

menuentry 'SchmitziOS' {
    search --no-floppy --fs-uuid --set=root $ROOT_UUID
    linux /boot/vmlinuz root=UUID=$ROOT_UUID ro quiet
}

menuentry 'SchmitziOS (Recovery Mode)' {
    search --no-floppy --fs-uuid --set=root $ROOT_UUID
    linux /boot/vmlinuz root=UUID=$ROOT_UUID ro single
}
GRUBEOF"

# Unmount chroot filesystems
umount /mnt/target/sys
umount /mnt/target/proc
umount /mnt/target/dev/pts
umount /mnt/target/dev

echo "==> Cleaning up..."
umount /mnt/target/boot
umount /mnt/target

echo ""
echo "========================================"
echo "      Installation Complete!"
echo "========================================"
echo ""
echo "SchmitziOS has been installed to $TARGET"
echo "You can now reboot and remove the installation media."
echo ""
read -p "Press Enter to continue..."
INSTALLEOF

chmod +x /root/iso/install.sh

## Step 8: Create GRUB Configuration for ISO

In [ ]:
cat > /root/iso/boot/grub/grub.cfg << 'GRUBEOF'
# SchmitziOS Live ISO GRUB Configuration

set default=0
set timeout=10

insmod all_video
insmod gfxterm
set gfxmode=auto
terminal_output gfxterm

set color_normal=white/black
set color_highlight=black/white

menuentry "SchmitziOS Live" {
    linux /boot/vmlinuz quiet
    initrd /boot/initramfs.img
}

menuentry "SchmitziOS Live (Verbose Boot)" {
    linux /boot/vmlinuz
    initrd /boot/initramfs.img
}

menuentry "SchmitziOS Installer" {
    linux /boot/vmlinuz quiet
    initrd /boot/initramfs.img
}
GRUBEOF

## Step 9: Create the Bootable ISO

In [ ]:
# Install GRUB modules for ISO
mkdir -p /root/iso/boot/grub/i386-pc
cp /usr/lib/grub/i386-pc/*.mod /root/iso/boot/grub/i386-pc/
cp /usr/lib/grub/i386-pc/*.lst /root/iso/boot/grub/i386-pc/

# Create BIOS boot image
grub-mkimage -O i386-pc -o /root/iso/boot/grub/i386-pc/core.img \
    -p /boot/grub \
    biosdisk iso9660 part_gpt part_msdos ext2 fat normal boot linux

# Combine with cdboot.img
cat /usr/lib/grub/i386-pc/cdboot.img /root/iso/boot/grub/i386-pc/core.img \
    > /root/iso/boot/grub/i386-pc/eltorito.img

# Create the ISO
cd /root
xorriso -as mkisofs \
    -iso-level 3 \
    -full-iso9660-filenames \
    -volid "SCHMITZIOS" \
    -eltorito-boot boot/grub/i386-pc/eltorito.img \
    -no-emul-boot \
    -boot-load-size 4 \
    -boot-info-table \
    -eltorito-catalog boot/grub/boot.cat \
    -output SchmitziOS-1.0.iso \
    iso/

echo ""
echo "========================================"
echo "      ISO Created Successfully!"
echo "========================================"
ls -lh /root/SchmitziOS-1.0.iso

## Using Your ISO

### Copy to another machine:
```bash
scp /root/SchmitziOS-1.0.iso user@otherpc:~/
```

### Write to USB drive:
```bash
# CAREFUL - use the correct device!
lsblk  # identify your USB drive
dd if=/root/SchmitziOS-1.0.iso of=/dev/sdX bs=4M status=progress conv=fsync
```

### Test in VirtualBox:
1. Create new VM
2. Attach SchmitziOS-1.0.iso as optical drive
3. Boot and verify live environment works
4. Run `./install.sh` to test installer

In [ ]:
## SSH Access to Your System

### Default SSH Credentials
Username: root
Password: password

⚠️ SECURITY WARNING: Change the default password immediately after first boot!

### Accessing via SSH

# 1. Boot SchmitziOS
#Boot your system (live ISO or installed version)

# 2. Find IP Address
#On the SchmitziOS console:
ip addr show
# Look for the IP address (e.g., 192.168.1.100)

# 3. Connect from Another Machine
#From your host or another computer on the same network:
ssh root@<IP_ADDRESS>
# Example: ssh root@192.168.1.100
# Password: password

# For VirtualBox with Port Forwarding
#If you set up port forwarding (Host Port 2222 → Guest Port 22):
ssh -p 2222 root@localhost

# Verify SSH is Running
# Check SSH daemon status
ps aux | grep sshd

# Start SSH manually if needed
/etc/rc.d/init.d/sshd start

# Check SSH is listening
netstat -tlnp | grep :22
# or
ss -tlnp | grep :22

# First Login Checklist
# After your first SSH login, immediately:

# Change root password:
   passwd root

# Create a regular user:
   useradd -m -s /bin/bash yourusername
   passwd yourusername

# Test network connectivity
   ping -c 3 google.com

# SSH Troubleshooting

# -- SSH Not Starting-- 
# Check for errors
/usr/sbin/sshd -d

# Check if config is valid
/usr/sbin/sshd -t

# Regenerate host keys if needed
ssh-keygen -A

# -- Can't Connect --
# Verify sshd is running
ps aux | grep sshd

# Check network is up
ip addr show
ping 8.8.8.8

# Check firewall (if configured)
iptables -L

# Password Authentication Fails
# Reset root password
passwd root

# Or use chpasswd method
echo 'root:newpassword' | chpasswd

# Verify password was set
grep root /etc/shadow

# Advanced: SSH Key Authentication

#For improved security, set up SSH key authentication:

#On your client machine:
# Generate SSH key pair (if you don't have one)
ssh-keygen -t ed25519 -C "your_email@example.com"

# Copy public key to SchmitziOS
ssh-copy-id root@<IP_ADDRESS>

# On SchmitziOS, disable password authentication (optional):
# Edit SSH config
sed -i 's/PasswordAuthentication yes/PasswordAuthentication no/' /etc/ssh/sshd_config

# Restart SSH
/etc/rc.d/init.d/sshd restart

## 🎉 Congratulations!

You now have a complete distributable SchmitziOS ISO that:

- ✅ Boots on BIOS systems
- ✅ Runs as a live system from RAM
- ✅ Includes a text-based installer
- ✅ Installs to any target disk with proper bootloader

You've built your own Linux distribution from scratch!

# .....Anyway

Now its time to restart and login. You'll need a bit of config to start the ethernet. This will be automatic eventually